# Scatterplot generator


In [ ]:

#from __future__ import annotations

import json
import math
import shutil
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import altair as alt
import kaleido
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except Exception:
    sns = None
import plotly.express as px
import plotly.graph_objects as go
import vl_convert



from groq import Groq
import os

# UNCOMMENT FOR GROQ



In [114]:
from pathlib import Path

# Project paths
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

#DATA_TRAIN = PROJECT / "data" / "train"

DATA_TRAIN = Path(r"C:\Users\Michelle\I2R\data\train")
# Normal generated outputs
OUT_ROOT = PROJECT / "outputs" / "generated" / "scatterplots"

# Testing outputs
SCATTER_TESTING_ROOT = PROJECT / "testing" / "scatter_testing"

OUT_ROOT.mkdir(parents=True, exist_ok=True)
SCATTER_TESTING_ROOT.mkdir(parents=True, exist_ok=True)

CLEAR_OUTPUT = True
LIBRARIES = ["altair", "matplotlib", "seaborn", "plotly"]

# Use metadata for consistency
SUBDIRS = ["images", "tables", "metadata"]

#REFERENCE_XLSX = DATA_REFERENCES / "scatter plots correct.xlsx"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Data


In [115]:
# ── Dataset registry ──────────────────────────────────────────────────────────
# Add new datasets here. Each entry needs:
#   "path"        : Path to the CSV file
#   "numeric_cols": List of numeric columns to use as x/y axes
#   "group_cols"  : List of categorical columns to use for grouping
#   "date_col"    : Optional date column name (None if not applicable)
#   "loader"      : Optional function name to pre-process the dataframe (or None)
DATASET_REGISTRY = {
    "warehouse_retail": {
        "path": DATA_TRAIN / "Warehouse_and_Retail_Sales.csv",
        "numeric_cols": ["RETAIL SALES", "WAREHOUSE SALES", "RETAIL TRANSFERS"],
        "group_cols":   ["ITEM TYPE", "SUPPLIER"],
        "date_col":     "date",
        "agg_cols": ["YEAR", "MONTH"],
        "loader":       "load_warehouse_retail",
    },
    # ── Add new datasets below ────────────────────────────────────────────────
    # "my_dataset": {
    #     "path":         DATA_TRAIN / "my_file.csv",
    #     "numeric_cols": ["col_a", "col_b", "col_c"],
    #     "group_cols":   ["category"],
    #     "date_col":     None,
    #     "loader":       None,
    # },

    "london_borough_sector_jobs": {
        "path": DATA_TRAIN / "london_borough_sector_jobs.csv",  # <- adjust filename if needed
        "numeric_cols": ["EMPLOYEE_JOBS"],
        "group_cols":   ["SECTOR", "BOROUGH"],
        "date_col":     None,
        "agg_cols":     ["YEAR"],
        "loader":       "load_london_borough_sector_jobs",
    },

}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [116]:
# ── Dataset loaders ───────────────────────────────────────────────────────────
# Add a loader function for each dataset that needs pre-processing.
# The function receives the raw DataFrame and returns a cleaned one.

def load_warehouse_retail(df: pd.DataFrame) -> pd.DataFrame:
    df["date"] = pd.to_datetime(
        dict(year=df["YEAR"].astype("Int64"), month=df["MONTH"].astype("Int64"), day=1),
        errors="coerce",
    )
    return df


def load_london_borough_sector_jobs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean London Datastore borough-by-sector employee jobs data.

    Handles the CSV format where the first rows are title/metadata rows and
    the real header row contains columns such as borough, sector, 1971, 1972, ...
    """
    import re

    df = df.copy()

    # Drop completely empty rows/cols first
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all").reset_index(drop=True)

    def is_year_like(value) -> bool:
        s = str(value).strip()
        s = re.sub(r"\.0$", "", s)
        return bool(re.fullmatch(r"(19|20)\d{2}", s))

    def normalise_year(value) -> str | None:
        s = str(value).strip()
        s = re.sub(r"\.0$", "", s)
        if is_year_like(s):
            return s
        return None

    # If pandas used the wrong header row, find the real header row
    current_year_cols = [c for c in df.columns if is_year_like(c)]

    if len(current_year_cols) < 5:
        header_row_idx = None
        best_year_count = 0

        for i in range(min(len(df), 30)):
            row_values = df.iloc[i].tolist()
            year_count = sum(is_year_like(v) for v in row_values)
            if year_count > best_year_count:
                best_year_count = year_count
                header_row_idx = i

        if header_row_idx is None or best_year_count < 5:
            raise ValueError(
                "Could not find the real header row containing year columns. "
                f"Columns found: {list(df.columns)}"
            )

        new_columns = df.iloc[header_row_idx].tolist()
        df = df.iloc[header_row_idx + 1:].copy()
        df.columns = new_columns

    # Clean column names
    cleaned_cols = []
    for c in df.columns:
        if pd.isna(c):
            cleaned_cols.append("")
        else:
            cleaned_cols.append(str(c).strip())

    df.columns = cleaned_cols
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all").reset_index(drop=True)

    # Normalise year column names
    renamed = {}
    for c in df.columns:
        y = normalise_year(c)
        if y is not None:
            renamed[c] = y
    df = df.rename(columns=renamed)

    year_cols = [c for c in df.columns if is_year_like(c)]
    if not year_cols:
        raise ValueError(
            "Could not find year columns after cleaning. "
            f"Columns found: {list(df.columns)}"
        )

    col_lookup = {str(c).lower().strip(): c for c in df.columns}

    def find_col(possible_names):
        for name in possible_names:
            key = name.lower().strip()
            if key in col_lookup:
                return col_lookup[key]
        return None

    borough_col = find_col([
        "borough", "borough name", "local authority", "local authority name",
        "area", "geography", "name"
    ])

    sector_col = find_col([
        "sector", "industry", "industry sector", "sector name", "sic section", "sic"
    ])

    non_year_cols = [c for c in df.columns if c not in year_cols and str(c).strip() != ""]

    if borough_col is None or sector_col is None:
        candidate_cols = []
        for c in non_year_cols:
            non_null = df[c].dropna().astype(str).str.strip()
            if len(non_null) == 0:
                continue
            unique_count = non_null.nunique()
            avg_len = non_null.str.len().mean()
            candidate_cols.append((c, unique_count, avg_len))

        candidate_cols = sorted(candidate_cols, key=lambda x: (x[1], x[2]), reverse=True)
        inferred_cols = [c for c, _, _ in candidate_cols]

        if borough_col is None and len(inferred_cols) >= 1:
            borough_col = inferred_cols[0]
        if sector_col is None and len(inferred_cols) >= 2:
            sector_col = inferred_cols[1]

    if borough_col is None or sector_col is None:
        raise ValueError(
            "Could not identify borough and sector columns. "
            f"Columns found: {list(df.columns)}"
        )

    # Wide -> long
    df = df.melt(
        id_vars=[borough_col, sector_col],
        value_vars=year_cols,
        var_name="YEAR",
        value_name="EMPLOYEE_JOBS",
    )

    df = df.rename(columns={
        borough_col: "BOROUGH",
        sector_col: "SECTOR",
    })

    df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce").astype("Int64")
    df["EMPLOYEE_JOBS"] = (
        df["EMPLOYEE_JOBS"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df["EMPLOYEE_JOBS"] = pd.to_numeric(df["EMPLOYEE_JOBS"], errors="coerce")

    df["BOROUGH"] = df["BOROUGH"].astype(str).str.strip()
    df["SECTOR"]  = df["SECTOR"].astype(str).str.strip()

    df = df.dropna(subset=["YEAR", "BOROUGH", "SECTOR", "EMPLOYEE_JOBS"])

    df = df[
        ~df["BOROUGH"].str.lower().isin([
            "nan", "", "total", "london", "greater london"
        ])
    ]

    df = df[
        ~df["SECTOR"].str.lower().isin([
            "nan", "", "total", "all", "all industries", "all sectors"
        ])
    ]

    return df.reset_index(drop=True)


def get_loader(name: str):
    loaders = {
        "load_warehouse_retail": load_warehouse_retail,
        "load_london_borough_sector_jobs": load_london_borough_sector_jobs,
    }
    return loaders.get(name)


def load_dataset(dataset_name: str) -> pd.DataFrame | None:
    """Load and pre-process a registered dataset. Returns None if file not found."""
    spec = DATASET_REGISTRY.get(dataset_name)
    if spec is None:
        raise KeyError(f"Unknown dataset: {dataset_name}")
    path = spec["path"]
    if not path.exists():
        print(f"Dataset not found: {path}")
        return None
    df = pd.read_csv(path)
    if spec["loader"]:
        loader_fn = get_loader(spec["loader"])
        if loader_fn:
            df = loader_fn(df)
    return df


# Load all available datasets
DATASETS = {}
for ds_name, ds_spec in DATASET_REGISTRY.items():
    ds = load_dataset(ds_name)
    if ds is not None:
        DATASETS[ds_name] = ds
        print(f"Loaded '{ds_name}': {len(ds)} rows, cols: {list(ds.columns)}")

# Pick default dataset (first available)
DEFAULT_DATASET = next(iter(DATASETS)) if DATASETS else None
print(f"Default dataset: {DEFAULT_DATASET}")

Loaded 'warehouse_retail': 307645 rows, cols: ['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES', 'date']
Loaded 'london_borough_sector_jobs': 28508 rows, cols: ['BOROUGH', 'SECTOR', 'YEAR', 'EMPLOYEE_JOBS']
Default dataset: warehouse_retail


In [117]:
# def sample_scatter_data_from_df(
#     df: pd.DataFrame,
#     spec: dict,
#     rng: np.random.Generator,
#     style: dict,
#     min_points: int = 3,
# ) -> tuple[pd.DataFrame, dict] | None:
#     numeric_cols = spec["numeric_cols"]
#     agg_cols     = spec.get("agg_cols")

#     if len(numeric_cols) < 2:
#         return None

#     x_col, y_col = rng.choice(numeric_cols, size=2, replace=False).tolist()

#     # Randomly choose sampling strategy
#     strategies = ["time"]
#     if "group_cols" in spec:
#         strategies.append("supplier")
#         strategies.append("item_type_month")
#     strategy = str(rng.choice(strategies, p=[0.40, 0.40, 0.20]))

#     if strategy == "time" and agg_cols and all(c in df.columns for c in agg_cols):
#         # Strategy 1: aggregate by YEAR+MONTH → 24 clean points
#         plot_df = df.groupby(agg_cols)[[x_col, y_col]].sum().reset_index()
#         plot_df["_label"] = plot_df[agg_cols].astype(str).agg("-".join, axis=1)
#         group_col = "_label"

#     elif strategy == "supplier":
#         # Strategy 2: top-N suppliers by total x_col sales
#         n_top = int(rng.choice([15, 20, 25, 30]))
#         gc = "SUPPLIER"
#         if gc not in df.columns:
#             return None
#         top_idx = df.groupby(gc)[x_col].sum().nlargest(n_top).index
#         plot_df = df[df[gc].isin(top_idx)].groupby(gc)[[x_col, y_col]].sum().reset_index()
#         plot_df["_label"] = plot_df[gc].astype(str)
#         group_col = "_label"

#     elif strategy == "item_type_month" and agg_cols and all(c in df.columns for c in agg_cols):
#         # Strategy 3: aggregate by ITEM TYPE + YEAR+MONTH, sample random subset
#         gc = "ITEM TYPE"
#         if gc not in df.columns:
#             return None
#         plot_df = df.groupby([gc] + agg_cols)[[x_col, y_col]].sum().reset_index()
#         plot_df["_label"] = plot_df[gc] + " " + plot_df[agg_cols].astype(str).agg("-".join, axis=1)
#         # Sample a random subset of 20-40 rows
#         n_sample = min(int(rng.choice([20, 30, 40])), len(plot_df))
#         plot_df = plot_df.sample(n=n_sample, random_state=int(rng.integers(0, 10000)))
#         group_col = "_label"

#     else:
#         # Fallback: aggregate by first group col
#         gc = str(rng.choice(spec["group_cols"]))
#         if gc not in df.columns:
#             return None
#         plot_df = df.groupby(gc)[[x_col, y_col]].sum().reset_index()
#         plot_df["_label"] = plot_df[gc].astype(str)
#         group_col = "_label"

#     plot_df = plot_df.dropna(subset=[x_col, y_col])
#     plot_df = plot_df[(plot_df[x_col] > 0) & (plot_df[y_col] > 0)]

#     if len(plot_df) < min_points:
#         return None

#     plot_df = plot_df.reset_index(drop=True)

#     # Apply n_groups grouping via quantile cut on x
#     n_groups = max(1, int(style.get("n_groups", 1)))
#     n_groups = min(n_groups, len(plot_df))
#     if n_groups > 1:
#         try:
#             plot_df["group"] = pd.qcut(
#                 plot_df[x_col], q=n_groups,
#                 labels=[f"Group {i+1}" for i in range(n_groups)],
#                 duplicates="drop",
#             ).astype(str)
#         except Exception:
#             plot_df["group"] = "Group 1"
#     else:
#         plot_df["group"] = "Group 1"

#     actual_groups = plot_df["group"].unique().tolist()
#     n_actual      = len(actual_groups)
#     n_colors = max(1, min(int(style.get("n_colors", 1)), n_actual))
#     n_shapes = max(1, min(int(style.get("n_shapes", 1)), n_actual))

#     color_cats = [f"Color {i+1}" for i in range(n_colors)]
#     shape_cats = [f"Shape {i+1}" for i in range(n_shapes)]
#     color_map  = {g: color_cats[i % n_colors] for i, g in enumerate(actual_groups)}
#     shape_map  = {g: shape_cats[i % n_shapes] for i, g in enumerate(actual_groups)}

#     plot_df["color_group"] = plot_df["group"].map(color_map)
#     plot_df["shape_group"] = plot_df["group"].map(shape_map)
#     plot_df["point_id"]    = plot_df[group_col].astype(str)
#     plot_df["label"]       = plot_df[group_col].astype(str)
#     plot_df["show_label"]  = False

#     label_code = int(style.get("direct_labels", 0))
#     if label_code == 1:
#         plot_df["show_label"] = True
#     elif label_code == 2:
#         n = min(max(3, int(round(len(plot_df) * 0.30))), len(plot_df))
#         idx = rng.choice(plot_df.index.to_numpy(), size=n, replace=False)
#         plot_df.loc[idx, "show_label"] = True

#     plot_df = plot_df.rename(columns={x_col: "x", y_col: "y"})

#     context = {
#         "x_label":    x_col.title(),
#         "y_label":    y_col.title(),
#         "x_col":      x_col,
#         "y_col":      y_col,
#         "group_col":  group_col,
#         "strategy":   strategy,
#         "n_points":   int(len(plot_df)),
#         "n_groups":   int(plot_df["group"].nunique()),
#         "orientation": int(style.get("scatter_orientation", 1)),
#         "group_mode":  group_col,
#         "x_range":    [float(plot_df["x"].min()), float(plot_df["x"].max())],
#         "y_range":    [float(plot_df["y"].min()), float(plot_df["y"].max())],
#     }

#     return plot_df, context

In [118]:
def sample_scatter_data_from_df(
    df: pd.DataFrame,
    spec: dict,
    rng: np.random.Generator,
    style: dict,
    min_points: int = 3,
) -> tuple[pd.DataFrame, dict] | None:
    """
    Build a scatter-plot-ready DataFrame from a real dataset.

    Output columns expected by renderers:
        x, y, group, color_group, shape_group, point_id, label, show_label
    """
    df = df.copy()

    numeric_cols = [c for c in spec.get("numeric_cols", []) if c in df.columns]
    group_cols   = [c for c in spec.get("group_cols", []) if c in df.columns]
    agg_cols     = [c for c in spec.get("agg_cols", []) if c in df.columns]

    plot_df = None
    context = None

    # ------------------------------------------------------------------
    # CASE A: General datasets with 2+ numeric columns
    # ------------------------------------------------------------------
    if len(numeric_cols) >= 2:
        x_col, y_col = rng.choice(numeric_cols, size=2, replace=False).tolist()

        candidate_strategies = []
        if agg_cols:
            candidate_strategies.append("time")
        if "SUPPLIER" in df.columns:
            candidate_strategies.append("supplier")
        if "ITEM TYPE" in df.columns and agg_cols:
            candidate_strategies.append("item_type_time")
        if group_cols:
            candidate_strategies.append("group")

        if not candidate_strategies:
            return None

        strategy = str(rng.choice(candidate_strategies))

        if strategy == "time":
            plot_df = df.groupby(agg_cols)[[x_col, y_col]].sum().reset_index()
            plot_df["_label"] = plot_df[agg_cols].astype(str).agg("-".join, axis=1)

        elif strategy == "supplier":
            n_top = int(rng.choice([15, 20, 25, 30]))
            gc = "SUPPLIER"
            top_idx = df.groupby(gc)[x_col].sum().nlargest(n_top).index
            plot_df = (
                df[df[gc].isin(top_idx)]
                .groupby(gc)[[x_col, y_col]]
                .sum()
                .reset_index()
            )
            plot_df["_label"] = plot_df[gc].astype(str)

        elif strategy == "item_type_time":
            gc = "ITEM TYPE"
            plot_df = df.groupby([gc] + agg_cols)[[x_col, y_col]].sum().reset_index()
            plot_df["_label"] = plot_df[gc].astype(str) + " " + plot_df[agg_cols].astype(str).agg("-".join, axis=1)
            n_sample = min(int(rng.choice([20, 30, 40])), len(plot_df))
            plot_df = plot_df.sample(n=n_sample, random_state=int(rng.integers(0, 10000)))

        else:  # generic grouped aggregate
            gc = str(rng.choice(group_cols))
            plot_df = df.groupby(gc)[[x_col, y_col]].sum().reset_index()
            plot_df["_label"] = plot_df[gc].astype(str)

        plot_df = plot_df.dropna(subset=[x_col, y_col])
        plot_df = plot_df[(plot_df[x_col] > 0) & (plot_df[y_col] > 0)]
        if len(plot_df) < min_points:
            return None

        plot_df = plot_df.rename(columns={x_col: "x", y_col: "y"})
        plot_df["point_id"] = plot_df["_label"].astype(str)
        plot_df["label"] = plot_df["_label"].astype(str)

        context = {
            "x_label": x_col.title(),
            "y_label": y_col.title(),
            "x_col": x_col,
            "y_col": y_col,
            "strategy": strategy,
        }

    # ------------------------------------------------------------------
    # CASE B: London jobs dataset (derived scatter strategies)
    # ------------------------------------------------------------------
    elif {"BOROUGH", "SECTOR", "YEAR", "EMPLOYEE_JOBS"}.issubset(df.columns):
        strategy = str(rng.choice(
            ["year_vs_jobs", "sector_vs_sector", "borough_vs_borough"],
            p=[0.50, 0.30, 0.20]
        ))

        # 1) YEAR vs EMPLOYEE_JOBS
        if strategy == "year_vs_jobs":
            series_col = str(rng.choice(["BOROUGH", "SECTOR"]))
            fixed_col = "SECTOR" if series_col == "BOROUGH" else "BOROUGH"

            # pick one fixed category with good coverage
            fixed_candidates = (
                df.groupby(fixed_col)["EMPLOYEE_JOBS"]
                .sum()
                .sort_values(ascending=False)
                .index
                .tolist()
            )
            if not fixed_candidates:
                return None

            fixed_val = fixed_candidates[int(rng.integers(0, min(8, len(fixed_candidates))))]

            sub = df[df[fixed_col] == fixed_val].copy()
            top_series = (
                sub.groupby(series_col)["EMPLOYEE_JOBS"]
                .sum()
                .nlargest(int(rng.choice([2, 3, 4])))
                .index
                .tolist()
            )
            if not top_series:
                return None

            sub = sub[sub[series_col].isin(top_series)].copy()
            sub = sub.dropna(subset=["YEAR", "EMPLOYEE_JOBS"])
            if len(sub) < min_points:
                return None

            plot_df = sub.rename(columns={"YEAR": "x", "EMPLOYEE_JOBS": "y"})
            plot_df["point_id"] = plot_df[series_col].astype(str) + " " + plot_df["x"].astype(str)
            plot_df["label"] = plot_df["point_id"].astype(str)
            plot_df["_label"] = plot_df["point_id"].astype(str)

            context = {
                "x_label": "Year",
                "y_label": "Employee Jobs",
                "x_col": "YEAR",
                "y_col": "EMPLOYEE_JOBS",
                "strategy": strategy,
                "detail": f"{series_col} within {fixed_col}={fixed_val}",
            }

        # 2) Compare two sectors across boroughs for one year
        elif strategy == "sector_vs_sector":
            year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
            if not year_candidates:
                return None
            year_val = int(rng.choice(year_candidates))

            sub = df[df["YEAR"] == year_val].copy()
            top_sectors = (
                sub.groupby("SECTOR")["EMPLOYEE_JOBS"]
                .sum()
                .nlargest(10)
                .index
                .tolist()
            )
            if len(top_sectors) < 2:
                return None

            sector_a, sector_b = rng.choice(top_sectors, size=2, replace=False).tolist()

            wide = (
                sub[sub["SECTOR"].isin([sector_a, sector_b])]
                .pivot_table(index="BOROUGH", columns="SECTOR", values="EMPLOYEE_JOBS", aggfunc="sum")
                .dropna()
                .reset_index()
            )
            if len(wide) < min_points:
                return None

            plot_df = wide.rename(columns={sector_a: "x", sector_b: "y"})
            plot_df["point_id"] = plot_df["BOROUGH"].astype(str)
            plot_df["label"] = plot_df["BOROUGH"].astype(str)
            plot_df["_label"] = plot_df["BOROUGH"].astype(str)

            context = {
                "x_label": f"{sector_a.title()} jobs",
                "y_label": f"{sector_b.title()} jobs",
                "x_col": sector_a,
                "y_col": sector_b,
                "strategy": strategy,
                "detail": f"Borough comparison in {year_val}",
            }

        # 3) Compare two boroughs across sectors for one year
        else:
            year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
            if not year_candidates:
                return None
            year_val = int(rng.choice(year_candidates))

            sub = df[df["YEAR"] == year_val].copy()
            top_boroughs = (
                sub.groupby("BOROUGH")["EMPLOYEE_JOBS"]
                .sum()
                .nlargest(10)
                .index
                .tolist()
            )
            if len(top_boroughs) < 2:
                return None

            borough_a, borough_b = rng.choice(top_boroughs, size=2, replace=False).tolist()

            wide = (
                sub[sub["BOROUGH"].isin([borough_a, borough_b])]
                .pivot_table(index="SECTOR", columns="BOROUGH", values="EMPLOYEE_JOBS", aggfunc="sum")
                .dropna()
                .reset_index()
            )
            if len(wide) < min_points:
                return None

            plot_df = wide.rename(columns={borough_a: "x", borough_b: "y"})
            plot_df["point_id"] = wide["SECTOR"].astype(str)
            plot_df["label"] = wide["SECTOR"].astype(str)
            plot_df["_label"] = wide["SECTOR"].astype(str)

            context = {
                "x_label": f"{borough_a.title()} jobs",
                "y_label": f"{borough_b.title()} jobs",
                "x_col": borough_a,
                "y_col": borough_b,
                "strategy": strategy,
                "detail": f"Sector comparison in {year_val}",
            }

        plot_df = plot_df.dropna(subset=["x", "y"])
        plot_df = plot_df[np.isfinite(plot_df["x"]) & np.isfinite(plot_df["y"])]
        plot_df = plot_df[(plot_df["x"] > 0) & (plot_df["y"] > 0)]

        if len(plot_df) < min_points:
            return None

    else:
        return None

    # ------------------------------------------------------------------
    # Common group/color/shape assignment
    # ------------------------------------------------------------------
    plot_df = plot_df.reset_index(drop=True)

    n_groups = max(1, int(style.get("n_groups", 1)))
    n_groups = min(n_groups, len(plot_df))

    if n_groups > 1:
        try:
            plot_df["group"] = pd.qcut(
                plot_df["x"],
                q=n_groups,
                labels=[f"Group {i+1}" for i in range(n_groups)],
                duplicates="drop",
            ).astype(str)
        except Exception:
            plot_df["group"] = "Group 1"
    else:
        plot_df["group"] = "Group 1"

    actual_groups = plot_df["group"].unique().tolist()
    n_actual = len(actual_groups)

    n_colors = max(1, min(int(style.get("n_colors", 1)), n_actual))
    n_shapes = max(1, min(int(style.get("n_shapes", 1)), n_actual))

    color_cats = [f"Color {i+1}" for i in range(n_colors)]
    shape_cats = [f"Shape {i+1}" for i in range(n_shapes)]

    color_map = {g: color_cats[i % n_colors] for i, g in enumerate(actual_groups)}
    shape_map = {g: shape_cats[i % n_shapes] for i, g in enumerate(actual_groups)}

    plot_df["color_group"] = plot_df["group"].map(color_map)
    plot_df["shape_group"] = plot_df["group"].map(shape_map)
    plot_df["show_label"] = False

    label_code = int(style.get("direct_labels", 0))
    if label_code == 1:
        plot_df["show_label"] = True
    elif label_code == 2:
        n = min(max(3, int(round(len(plot_df) * 0.30))), len(plot_df))
        idx = rng.choice(plot_df.index.to_numpy(), size=n, replace=False)
        plot_df.loc[idx, "show_label"] = True

    context.update({
        "n_points": int(len(plot_df)),
        "n_groups": int(plot_df["group"].nunique()),
        "orientation": int(style.get("scatter_orientation", 1)),
        "x_range": [float(plot_df["x"].min()), float(plot_df["x"].max())],
        "y_range": [float(plot_df["y"].min()), float(plot_df["y"].max())],
    })

    return plot_df, context

In [119]:
# ── Load all registered datasets ──────────────────────────────────────────────
def get_loader(name: str):
    loaders = {
        "load_warehouse_retail":           load_warehouse_retail,
        "load_london_borough_sector_jobs": load_london_borough_sector_jobs,
    }
    return loaders.get(name)


def load_dataset(dataset_name: str) -> pd.DataFrame | None:
    """Load and pre-process a registered dataset. Returns None if file not found."""
    spec = DATASET_REGISTRY.get(dataset_name)
    if spec is None:
        raise KeyError(f"Unknown dataset: {dataset_name}")
    path = spec["path"]
    if not path.exists():
        print(f"Dataset not found: {path}")
        return None
    df = pd.read_csv(path)
    if spec["loader"]:
        loader_fn = get_loader(spec["loader"])
        if loader_fn:
            df = loader_fn(df)
    return df


DATASETS = {}
for ds_name, ds_spec in DATASET_REGISTRY.items():
    ds = load_dataset(ds_name)
    if ds is not None:
        DATASETS[ds_name] = ds
        print(f"Loaded '{ds_name}': {len(ds)} rows, cols: {list(ds.columns)}")

DEFAULT_DATASET = next(iter(DATASETS)) if DATASETS else None
print(f"Default dataset: {DEFAULT_DATASET}")

Loaded 'warehouse_retail': 307645 rows, cols: ['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES', 'date']
Loaded 'london_borough_sector_jobs': 28508 rows, cols: ['BOROUGH', 'SECTOR', 'YEAR', 'EMPLOYEE_JOBS']
Default dataset: warehouse_retail


## 1. Sampling weights

Weights are derived directly from observed frequencies in the reference dataset.
Each parameter maps `numeric_code -> probability`. No name-mapping layer -- renderers interpret codes directly.


In [120]:
# Sampling weights derived from observed dataset frequencies.
# Each key is a parameter name; values are {numeric_code: probability}.
# Codes match the column coding scheme in the reference Excel file.
SAMPLING_WEIGHTS = {
    "title_present": {
        "0": 0.6,
        "1": 0.4
    },
    "title_location": {
        "0": 0.62,
        "1": 0.24,
        "2": 0.14
    },
    "title_color": {
        "0": 0.23,
        "1": 0.12,
        "3": 0.03,
        "4": 0.6,
        "6": 0.02
    },
    "title_size": {
        "0": 0.2929,
        "1": 0.0606,
        "2": 0.0505,
        "3": 0.596
    },
    "subtitle_present": {
        "0": 0.95,
        "1": 0.05
    },
    "legend_present": {
        "0": 0.68,
        "1": 0.32
    },
    "legend_title_size": {
        "0": 0.82,
        "1": 0.18
    },
    "legend_title_color": {
        "0": 0.83,
        "1": 0.17
    },
    "legend_text_color": {
        "0": 0.22,
        "1": 0.07,
        "2": 0.68,
        "3": 0.02,
        "4": 0.01
    },
    "legend_outline": {
        "0": 0.84,
        "1": 0.16
    },
    "legend_fill": {
        "0": 0.71,
        "1": 0.04,
        "2": 0.13,
        "3": 0.12
    },
    "legend_orientation": {
        "0": 0.68,
        "2": 0.11,
        "3": 0.01,
        "4": 0.05,
        "5": 0.1,
        "6": 0.05
    },
    "direct_labels": {
        "0": 0.97,
        "1": 0.01,
        "2": 0.02
    },
    "label_content": {
        "0": 0.97,
        "1": 0.03
    },
    "label_color": {
        "0": 0.97,
        "1": 0.02,
        "2": 0.01
    },
    "chart_outline": {
        "0": 0.09,
        "1": 0.56,
        "2": 0.3,
        "3": 0.05
    },
    "gridlines": {
        "0": 0.45,
        "2": 0.09,
        "3": 0.46
    },
    "gridline_color": {
        "0": 0.45,
        "1": 0.37,
        "2": 0.08,
        "3": 0.07,
        "4": 0.03
    },
    "image_outline": {
        "0": 0.88,
        "1": 0.12
    },
    "background": {
        "0": 0.07,
        "1": 0.74,
        "2": 0.08,
        "3": 0.02,
        "4": 0.02,
        "5": 0.05,
        "6": 0.02
    },
    "axis_text_orientation": {
        "0": 0.76,
        "1": 0.02,
        "2": 0.19,
        "3": 0.02,
        "4": 0.01
    },
    "axis_text_color": {
        "0": 0.79,
        "1": 0.03,
        "2": 0.01,
        "3": 0.14,
        "4": 0.02,
        "5": 0.01
    },
    "axes_start_at_zero": {
        "0": 0.43,
        "1": 0.53,
        "2": 0.01,
        "3": 0.02,
        "4": 0.01
    },
    "x_scale": {
        "0": 0.07,
        "1": 0.14,
        "2": 0.06,
        "3": 0.14,
        "4": 0.02,
        "5": 0.12,
        "6": 0.04,
        "7": 0.01,
        "8": 0.01,
        "9": 0.1,
        "10": 0.03,
        "11": 0.02,
        "12": 0.01,
        "13": 0.01,
        "14": 0.01,
        "15": 0.03,
        "16": 0.01,
        "17": 0.03,
        "18": 0.07,
        "19": 0.02,
        "20": 0.01,
        "21": 0.01,
        "22": 0.01,
        "23": 0.02
    },
    "x_tick_step": {
        "0": 0.18,
        "1": 0.11,
        "2": 0.05,
        "3": 0.12,
        "4": 0.02,
        "5": 0.04,
        "6": 0.08,
        "7": 0.01,
        "8": 0.01,
        "9": 0.07,
        "10": 0.04,
        "11": 0.01,
        "12": 0.02,
        "13": 0.05,
        "14": 0.02,
        "15": 0.01,
        "16": 0.05,
        "17": 0.02,
        "18": 0.01,
        "19": 0.01,
        "20": 0.02,
        "21": 0.01,
        "22": 0.01,
        "23": 0.01,
        "24": 0.01,
        "25": 0.01
    },
    "y_scale": {
        "0": 0.19,
        "1": 0.16,
        "2": 0.03,
        "3": 0.12,
        "4": 0.03,
        "5": 0.09,
        "6": 0.01,
        "7": 0.03,
        "8": 0.04,
        "9": 0.01,
        "10": 0.01,
        "11": 0.04,
        "12": 0.07,
        "13": 0.02,
        "14": 0.02,
        "15": 0.01,
        "16": 0.02,
        "17": 0.01,
        "18": 0.03,
        "19": 0.01,
        "20": 0.02,
        "21": 0.01,
        "22": 0.02
    },
    "y_tick_step": {
        "0": 0.1,
        "1": 0.17,
        "2": 0.05,
        "3": 0.07,
        "4": 0.03,
        "5": 0.09,
        "6": 0.03,
        "7": 0.07,
        "8": 0.15,
        "9": 0.01,
        "10": 0.02,
        "11": 0.01,
        "12": 0.05,
        "13": 0.02,
        "14": 0.02,
        "15": 0.02,
        "16": 0.01,
        "17": 0.01,
        "18": 0.02,
        "19": 0.01,
        "20": 0.01,
        "21": 0.02,
        "22": 0.01
    },
    "scatter_orientation": {
        "0": 0.12,
        "1": 0.39,
        "2": 0.31,
        "3": 0.1,
        "4": 0.02,
        "5": 0.02,
        "6": 0.04
    },
    "n_groups": {
        "1": 0.6186,
        "2": 0.134,
        "3": 0.134,
        "4": 0.0412,
        "5": 0.0206,
        "6": 0.0206,
        "7": 0.0103,
        "8": 0.0103,
        "10": 0.0103
    },
    "point_shape_mode": {
        "0": 0.62,
        "1": 0.07,
        "2": 0.08,
        "3": 0.1,
        "4": 0.03,
        "5": 0.04,
        "6": 0.03,
        "7": 0.02,
        "8": 0.01
    },
    "n_shapes": {
        "0": 0.03,
        "1": 0.84,
        "2": 0.04,
        "3": 0.05,
        "4": 0.02,
        "5": 0.01,
        "6": 0.01
    },
    "n_colors": {
        "0": 0.02,
        "1": 0.58,
        "2": 0.13,
        "3": 0.15,
        "4": 0.03,
        "5": 0.03,
        "6": 0.03,
        "7": 0.01,
        "8": 0.01,
        "10": 0.01
    },
    "palette_type": {
        "0": 0.04,
        "1": 0.01,
        "2": 0.27,
        "3": 0.09,
        "4": 0.16,
        "5": 0.01,
        "6": 0.04,
        "7": 0.04,
        "8": 0.1,
        "9": 0.07,
        "10": 0.01,
        "11": 0.01,
        "12": 0.03,
        "13": 0.04,
        "14": 0.02,
        "15": 0.01,
        "16": 0.05
    },
    "n_points_bin": {
        "0": 0.02,
        "1": 0.04,
        "2": 0.16,
        "3": 0.77,
        "6": 0.01
    },
    "minor_ticks_x": {
        "0": 0.88,
        "1": 0.12
    },
    "minor_ticks_y": {
        "0": 0.86,
        "1": 0.12,
        "2": 0.02
    },
    "plot_orientation": {
        "0": 0.64,
        "1": 0.29,
        "2": 0.07
    }
}


In [121]:

def load_weights_from_excel(xlsx_path: Path, sheet: str = "ToFill_scatter") -> dict:
    """Recompute sampling weights live from the reference Excel file.
    
    Returns a dict matching the structure of SAMPLING_WEIGHTS above.
    Call this instead of using the hardcoded SAMPLING_WEIGHTS if you have
    an updated reference file.
    """
    PARAM_NAMES = [
        "title_present", "title_location", "title_color", "title_size",
        "subtitle_present", "legend_present", "legend_title_size",
        "legend_title_color", "legend_text_color", "legend_outline",
        "legend_fill", "legend_orientation", "direct_labels", "label_content",
        "label_color", "chart_outline", "gridlines", "gridline_color",
        "image_outline", "background", "axis_text_orientation",
        "axis_text_color", "axes_start_at_zero", "x_scale", "x_tick_step",
        "y_scale", "y_tick_step", "scatter_orientation", "n_groups",
        "point_shape_mode", "n_shapes", "n_colors", "palette_type",
        "n_points_bin", "minor_ticks_x", "minor_ticks_y", "plot_orientation",
    ]
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet, header=None)
    df = pd.read_excel(xlsx_path, sheet_name=sheet, header=1)
    data = df[df.iloc[:, 0].astype(str).str.match(r"scatter_\d+")]
    numeric_data = data.iloc[:, 1:len(PARAM_NAMES)+1].apply(pd.to_numeric, errors="coerce")

    weights = {}
    for i, col in enumerate(numeric_data.columns[:len(PARAM_NAMES)]):
        vals = numeric_data[col].dropna()
        counts = vals.value_counts().sort_index()
        total = len(vals)
        if total > 0:
            weights[PARAM_NAMES[i]] = {int(k): round(v / total, 6)
                                        for k, v in counts.items()}
    return weights


def normalize_weights(weights: dict) -> dict:
    """Ensure each parameter's weights sum to 1.0."""
    out = {}
    for param, codes in weights.items():
        total = sum(codes.values())
        out[param] = {int(k): v / total for k, v in codes.items()}
    return out


def sample_style(rng: np.random.Generator, weights: dict) -> dict:
    style = {}
    for param, code_probs in weights.items():
        codes = list(code_probs.keys())
        probs = np.array(list(code_probs.values()), dtype=float)
        probs /= probs.sum()
        style[param] = int(rng.choice(codes, p=probs))
    return style

### Validate sampling weights

This cell normalizes the observed fractions before generation. It also displays a quick check so small rounding differences do not affect sampling.


In [122]:
# Validate and normalize observed sampling weights.
# Use OBSERVED_WEIGHTS in generation cells; it is normalized to handle tiny rounding differences.
OBSERVED_WEIGHTS = normalize_weights(SAMPLING_WEIGHTS)

weight_check = pd.DataFrame([
    {
        "parameter": param,
        "n_codes": len(codes),
        "raw_sum": round(sum(codes.values()), 10),
        "normalized_sum": round(sum(OBSERVED_WEIGHTS[param].values()), 10),
    }
    for param, codes in SAMPLING_WEIGHTS.items()
])

weight_check

,parameter,n_codes,raw_sum,normalized_sum
0,title_present,2,1.0000,1.0
1,title_location,3,1.0000,1.0
2,title_color,5,1.0000,1.0
3,title_size,4,1.0000,1.0
4,subtitle_present,2,1.0000,1.0
5,legend_present,2,1.0000,1.0
6,legend_title_size,2,1.0000,1.0
7,legend_title_color,2,1.0000,1.0
8,legend_text_color,5,1.0000,1.0
9,legend_outline,2,1.0000,1.0


## 2. Scatter data sampling


In [123]:
# X scale bounds: code -> (min, max)
X_SCALE_BOUNDS = {
    0: (0, 20), 1: (0, 50), 2: (0, 5), 3: (0, 250), 4: (0, 5000),
    5: (0, 10), 6: (0, 100000), 7: (1900, 1920), 8: (0, 100),  # 8=no scale fallback
    9: (0, 100), 10: (0, 1000), 11: (0, 300000), 12: (0, 50000),
    13: (2000, 2010), 14: (0, 10000), 15: (0, 400), 16: (1990, 2010),
    17: (0, 2), 18: (0, 1), 19: (0, 10000), 20: (0, 365),
    21: (0, 100), 22: (-2000000, 2000000), 23: (-10000, 10000),
}

# Y scale bounds: code -> (min, max)
Y_SCALE_BOUNDS = {
    0: (0, 100), 1: (0, 50), 2: (0, 1000), 3: (0, 10), 4: (0, 6000),
    5: (0, 5), 6: (0, 100),  # 6=no scale fallback
    7: (0, 1), 8: (0, 20), 9: (-6, 8), 10: (0, 600), 11: (0, 300),
    12: (0, 350000), 13: (-30, 50), 14: (0, 250), 15: (-20, 0),
    16: (0, 4000), 17: (0, 8000000), 18: (0, 20000), 19: (-1e9, 1e9),
    20: (0, 60000), 21: (0, 200), 22: (0, 60),
}

# Palettes: code -> list of hex colours
PALETTES = {
    0:  ["#111111", "#333333", "#555555", "#777777", "#999999"],           # black
    1:  ["#1d3557", "#457b9d", "#264653", "#3a5a40", "#6c757d"],           # dark
    2:  ["#023e8a", "#0077b6", "#0096c7", "#00b4d8", "#48cae4"],           # dark blue
    3:  ["#1b4332", "#2d6a4f", "#40916c", "#52b788", "#74c69d"],           # dark green
    4:  ["#e63946", "#f4a261", "#ffbe0b", "#06d6a0", "#118ab2", "#8338ec"],# colorful
    5:  ["#b7e4c7", "#95d5b2", "#74c69d", "#52b788", "#40916c"],           # light green
    6:  ["#e63946", "#118ab2", "#ef233c", "#4361ee"],                      # red & blue
    7:  ["#a8dadc", "#ffd6a5", "#caffbf", "#fdffb6", "#c8b6ff"],           # light
    8:  ["#e63946", "#2a9d8f", "#264653"],                                 # RGB
    9:  ["#023e8a", "#e76f51", "#219ebc", "#fb8500"],                      # blue & orange
    10: ["#f4a261", "#118ab2", "#e63946", "#2a9d8f"],                      # orange/blue/red/green
    11: ["#118ab2", "#f4a261", "#2a9d8f"],                                 # blue/orange/green
    12: ["#ffb347", "#ffa500", "#ff8c00", "#e07b00"],                      # light orange
    13: ["#e76f51", "#f4a261", "#e9c46a", "#e07b00"],                      # orange
    14: ["#a8dadc", "#457b9d", "#1d3557", "#48cae4"],                      # light blue
    15: ["#f72585", "#b5179e", "#7209b7", "#560bad"],                      # pink shades
    16: ["#e63946", "#c1121f", "#9d0208", "#6a040f"],                      # red
}

# Altair shape names: code -> shape string
ALTAIR_SHAPES = {
    0: "circle", 1: "square", 2: "circle",       # by_group uses circle default
    3: "circle", 4: "circle", 5: "circle",
    6: "square", 7: "triangle-up", 8: "cross",
}

# Matplotlib markers: code -> marker string
MPL_MARKERS = {
    0: "o", 1: "s", 2: "o", 3: "o", 4: "o",
    5: "o", 6: "^", 7: "^", 8: "+",
}

# Plotly symbols: code -> symbol string
PLOTLY_SYMBOLS = {
    0: "circle", 1: "square", 2: "circle", 3: "circle-open",
    4: "circle-open", 5: "circle", 6: "square", 7: "triangle-up", 8: "cross",
}

# def sample_scatter_data_from_df(
#     df: pd.DataFrame,
#     spec: dict,
#     rng: np.random.Generator,
#     style: dict,
#     min_points: int = 3,
# ) -> tuple[pd.DataFrame, dict] | None:
#     """
#     Build a scatter-plot-ready DataFrame from a real dataset.

#     Output columns expected by renderers:
#         x, y, group, color_group, shape_group, point_id, label, show_label
#     """
#     df = df.copy()

#     numeric_cols = [c for c in spec.get("numeric_cols", []) if c in df.columns]
#     group_cols   = [c for c in spec.get("group_cols", []) if c in df.columns]
#     agg_cols     = [c for c in spec.get("agg_cols", []) if c in df.columns]

#     plot_df = None
#     context = None

#     # ------------------------------------------------------------------
#     # CASE A: General datasets with 2+ numeric columns
#     # ------------------------------------------------------------------
#     if len(numeric_cols) >= 2:
#         x_col, y_col = rng.choice(numeric_cols, size=2, replace=False).tolist()

#         candidate_strategies = []
#         if agg_cols:
#             candidate_strategies.append("time")
#         if "SUPPLIER" in df.columns:
#             candidate_strategies.append("supplier")
#         if "ITEM TYPE" in df.columns and agg_cols:
#             candidate_strategies.append("item_type_time")
#         if group_cols:
#             candidate_strategies.append("group")

#         if not candidate_strategies:
#             return None

#         strategy = str(rng.choice(candidate_strategies))

#         if strategy == "time":
#             plot_df = df.groupby(agg_cols)[[x_col, y_col]].sum().reset_index()
#             plot_df["_label"] = plot_df[agg_cols].astype(str).agg("-".join, axis=1)

#         elif strategy == "supplier":
#             n_top = int(rng.choice([15, 20, 25, 30]))
#             gc = "SUPPLIER"
#             top_idx = df.groupby(gc)[x_col].sum().nlargest(n_top).index
#             plot_df = (
#                 df[df[gc].isin(top_idx)]
#                 .groupby(gc)[[x_col, y_col]]
#                 .sum()
#                 .reset_index()
#             )
#             plot_df["_label"] = plot_df[gc].astype(str)

#         elif strategy == "item_type_time":
#             gc = "ITEM TYPE"
#             plot_df = df.groupby([gc] + agg_cols)[[x_col, y_col]].sum().reset_index()
#             plot_df["_label"] = plot_df[gc].astype(str) + " " + plot_df[agg_cols].astype(str).agg("-".join, axis=1)
#             n_sample = min(int(rng.choice([20, 30, 40])), len(plot_df))
#             plot_df = plot_df.sample(n=n_sample, random_state=int(rng.integers(0, 10000)))

#         else:  # generic grouped aggregate
#             gc = str(rng.choice(group_cols))
#             plot_df = df.groupby(gc)[[x_col, y_col]].sum().reset_index()
#             plot_df["_label"] = plot_df[gc].astype(str)

#         plot_df = plot_df.dropna(subset=[x_col, y_col])
#         plot_df = plot_df[(plot_df[x_col] > 0) & (plot_df[y_col] > 0)]
#         if len(plot_df) < min_points:
#             return None

#         plot_df = plot_df.rename(columns={x_col: "x", y_col: "y"})
#         plot_df["point_id"] = plot_df["_label"].astype(str)
#         plot_df["label"] = plot_df["_label"].astype(str)

#         context = {
#             "x_label": x_col.title(),
#             "y_label": y_col.title(),
#             "x_col": x_col,
#             "y_col": y_col,
#             "strategy": strategy,
#         }

#     # ------------------------------------------------------------------
#     # CASE B: London jobs dataset (derived scatter strategies)
#     # ------------------------------------------------------------------
#     elif {"BOROUGH", "SECTOR", "YEAR", "EMPLOYEE_JOBS"}.issubset(df.columns):
#         strategy = str(rng.choice(
#             ["year_vs_jobs", "sector_vs_sector", "borough_vs_borough"],
#             p=[0.50, 0.30, 0.20]
#         ))

#         # 1) YEAR vs EMPLOYEE_JOBS
#         if strategy == "year_vs_jobs":
#             series_col = str(rng.choice(["BOROUGH", "SECTOR"]))
#             fixed_col = "SECTOR" if series_col == "BOROUGH" else "BOROUGH"

#             # pick one fixed category with good coverage
#             fixed_candidates = (
#                 df.groupby(fixed_col)["EMPLOYEE_JOBS"]
#                 .sum()
#                 .sort_values(ascending=False)
#                 .index
#                 .tolist()
#             )
#             if not fixed_candidates:
#                 return None

#             fixed_val = fixed_candidates[int(rng.integers(0, min(8, len(fixed_candidates))))]

#             sub = df[df[fixed_col] == fixed_val].copy()
#             top_series = (
#                 sub.groupby(series_col)["EMPLOYEE_JOBS"]
#                 .sum()
#                 .nlargest(int(rng.choice([2, 3, 4])))
#                 .index
#                 .tolist()
#             )
#             if not top_series:
#                 return None

#             sub = sub[sub[series_col].isin(top_series)].copy()
#             sub = sub.dropna(subset=["YEAR", "EMPLOYEE_JOBS"])
#             if len(sub) < min_points:
#                 return None

#             plot_df = sub.rename(columns={"YEAR": "x", "EMPLOYEE_JOBS": "y"})
#             plot_df["point_id"] = plot_df[series_col].astype(str) + " " + plot_df["x"].astype(str)
#             plot_df["label"] = plot_df["point_id"].astype(str)
#             plot_df["_label"] = plot_df["point_id"].astype(str)

#             context = {
#                 "x_label": "Year",
#                 "y_label": "Employee Jobs",
#                 "x_col": "YEAR",
#                 "y_col": "EMPLOYEE_JOBS",
#                 "strategy": strategy,
#                 "detail": f"{series_col} within {fixed_col}={fixed_val}",
#             }

#         # 2) Compare two sectors across boroughs for one year
#         elif strategy == "sector_vs_sector":
#             year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
#             if not year_candidates:
#                 return None
#             year_val = int(rng.choice(year_candidates))

#             sub = df[df["YEAR"] == year_val].copy()
#             top_sectors = (
#                 sub.groupby("SECTOR")["EMPLOYEE_JOBS"]
#                 .sum()
#                 .nlargest(10)
#                 .index
#                 .tolist()
#             )
#             if len(top_sectors) < 2:
#                 return None

#             sector_a, sector_b = rng.choice(top_sectors, size=2, replace=False).tolist()

#             wide = (
#                 sub[sub["SECTOR"].isin([sector_a, sector_b])]
#                 .pivot_table(index="BOROUGH", columns="SECTOR", values="EMPLOYEE_JOBS", aggfunc="sum")
#                 .dropna()
#                 .reset_index()
#             )
#             if len(wide) < min_points:
#                 return None

#             plot_df = wide.rename(columns={sector_a: "x", sector_b: "y"})
#             plot_df["point_id"] = plot_df["BOROUGH"].astype(str)
#             plot_df["label"] = plot_df["BOROUGH"].astype(str)
#             plot_df["_label"] = plot_df["BOROUGH"].astype(str)

#             context = {
#                 "x_label": f"{sector_a.title()} jobs",
#                 "y_label": f"{sector_b.title()} jobs",
#                 "x_col": sector_a,
#                 "y_col": sector_b,
#                 "strategy": strategy,
#                 "detail": f"Borough comparison in {year_val}",
#             }

#         # 3) Compare two boroughs across sectors for one year
#         else:
#             year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
#             if not year_candidates:
#                 return None
#             year_val = int(rng.choice(year_candidates))

#             sub = df[df["YEAR"] == year_val].copy()
#             top_boroughs = (
#                 sub.groupby("BOROUGH")["EMPLOYEE_JOBS"]
#                 .sum()
#                 .nlargest(10)
#                 .index
#                 .tolist()
#             )
#             if len(top_boroughs) < 2:
#                 return None

#             borough_a, borough_b = rng.choice(top_boroughs, size=2, replace=False).tolist()

#             wide = (
#                 sub[sub["BOROUGH"].isin([borough_a, borough_b])]
#                 .pivot_table(index="SECTOR", columns="BOROUGH", values="EMPLOYEE_JOBS", aggfunc="sum")
#                 .dropna()
#                 .reset_index()
#             )
#             if len(wide) < min_points:
#                 return None

#             plot_df = wide.rename(columns={borough_a: "x", borough_b: "y"})
#             plot_df["point_id"] = wide["SECTOR"].astype(str)
#             plot_df["label"] = wide["SECTOR"].astype(str)
#             plot_df["_label"] = wide["SECTOR"].astype(str)

#             context = {
#                 "x_label": f"{borough_a.title()} jobs",
#                 "y_label": f"{borough_b.title()} jobs",
#                 "x_col": borough_a,
#                 "y_col": borough_b,
#                 "strategy": strategy,
#                 "detail": f"Sector comparison in {year_val}",
#             }

#         plot_df = plot_df.dropna(subset=["x", "y"])
#         plot_df = plot_df[np.isfinite(plot_df["x"]) & np.isfinite(plot_df["y"])]
#         plot_df = plot_df[(plot_df["x"] > 0) & (plot_df["y"] > 0)]

#         if len(plot_df) < min_points:
#             return None

#     else:
#         return None

#     # ------------------------------------------------------------------
#     # Common group/color/shape assignment
#     # ------------------------------------------------------------------
#     plot_df = plot_df.reset_index(drop=True)

#     n_groups = max(1, int(style.get("n_groups", 1)))
#     n_groups = min(n_groups, len(plot_df))

#     if n_groups > 1:
#         try:
#             plot_df["group"] = pd.qcut(
#                 plot_df["x"],
#                 q=n_groups,
#                 labels=[f"Group {i+1}" for i in range(n_groups)],
#                 duplicates="drop",
#             ).astype(str)
#         except Exception:
#             plot_df["group"] = "Group 1"
#     else:
#         plot_df["group"] = "Group 1"

#     actual_groups = plot_df["group"].unique().tolist()
#     n_actual = len(actual_groups)

#     n_colors = max(1, min(int(style.get("n_colors", 1)), n_actual))
#     n_shapes = max(1, min(int(style.get("n_shapes", 1)), n_actual))

#     color_cats = [f"Color {i+1}" for i in range(n_colors)]
#     shape_cats = [f"Shape {i+1}" for i in range(n_shapes)]

#     color_map = {g: color_cats[i % n_colors] for i, g in enumerate(actual_groups)}
#     shape_map = {g: shape_cats[i % n_shapes] for i, g in enumerate(actual_groups)}

#     plot_df["color_group"] = plot_df["group"].map(color_map)
#     plot_df["shape_group"] = plot_df["group"].map(shape_map)
#     plot_df["show_label"] = False

#     label_code = int(style.get("direct_labels", 0))
#     if label_code == 1:
#         plot_df["show_label"] = True
#     elif label_code == 2:
#         n = min(max(3, int(round(len(plot_df) * 0.30))), len(plot_df))
#         idx = rng.choice(plot_df.index.to_numpy(), size=n, replace=False)
#         plot_df.loc[idx, "show_label"] = True

#     context.update({
#         "n_points": int(len(plot_df)),
#         "n_groups": int(plot_df["group"].nunique()),
#         "orientation": int(style.get("scatter_orientation", 1)),
#         "x_range": [float(plot_df["x"].min()), float(plot_df["x"].max())],
#         "y_range": [float(plot_df["y"].min()), float(plot_df["y"].max())],
#     })

#     return plot_df, context




# data sampling

In [124]:
def sample_scatter_data_from_df(
    df: pd.DataFrame,
    spec: dict,
    rng: np.random.Generator,
    style: dict,
    min_points: int = 3,
) -> tuple[pd.DataFrame, dict] | None:
    """
    Build a scatter-plot-ready DataFrame from a real dataset.

    Output columns expected by renderers:
        x, y, group, color_group, shape_group, point_id, label, show_label
    """
    df = df.copy()

    numeric_cols = [c for c in spec.get("numeric_cols", []) if c in df.columns]
    group_cols   = [c for c in spec.get("group_cols", []) if c in df.columns]
    agg_cols     = [c for c in spec.get("agg_cols", []) if c in df.columns]

    plot_df = None
    context = None

    # ------------------------------------------------------------------
    # CASE A: General datasets with 2+ numeric columns
    # ------------------------------------------------------------------
    if len(numeric_cols) >= 2:
        x_col, y_col = rng.choice(numeric_cols, size=2, replace=False).tolist()

        candidate_strategies = []
        if "SUPPLIER" in df.columns:
            candidate_strategies.append("supplier")
        if "ITEM TYPE" in df.columns and agg_cols:
            candidate_strategies.append("item_type_time")
        if group_cols:
            candidate_strategies.append("group")

        if not candidate_strategies:
            return None

        strategy = str(rng.choice(candidate_strategies))

        if strategy == "supplier":
            n_top = int(rng.choice([15, 20, 25, 30]))
            gc = "SUPPLIER"
            top_idx = df.groupby(gc)[x_col].sum().nlargest(n_top).index
            plot_df = (
                df[df[gc].isin(top_idx)]
                .groupby(gc)[[x_col, y_col]]
                .sum()
                .reset_index()
            )
            plot_df["_label"] = plot_df[gc].astype(str)

        elif strategy == "item_type_time":
            gc = "ITEM TYPE"
            plot_df = df.groupby([gc] + agg_cols)[[x_col, y_col]].sum().reset_index()
            plot_df["_label"] = plot_df[gc].astype(str) + " " + plot_df[agg_cols].astype(str).agg("-".join, axis=1)
            n_sample = min(int(rng.choice([20, 30, 40])), len(plot_df))
            plot_df = plot_df.sample(n=n_sample, random_state=int(rng.integers(0, 10000)))

        else:  # generic grouped aggregate
            gc = str(rng.choice(group_cols))
            plot_df = df.groupby(gc)[[x_col, y_col]].sum().reset_index()
            plot_df["_label"] = plot_df[gc].astype(str)

        plot_df = plot_df.dropna(subset=[x_col, y_col])
        plot_df = plot_df[(plot_df[x_col] > 0) & (plot_df[y_col] > 0)]
        if len(plot_df) < min_points:
            return None

        plot_df = plot_df.rename(columns={x_col: "x", y_col: "y"})
        plot_df["point_id"] = plot_df["_label"].astype(str)
        plot_df["label"] = plot_df["_label"].astype(str)

        context = {
            "x_label": x_col.title(),
            "y_label": y_col.title(),
            "x_col": x_col,
            "y_col": y_col,
            "strategy": strategy,
        }

    # ------------------------------------------------------------------
    # CASE B: London jobs dataset — genuine scatter strategies only
    # ------------------------------------------------------------------
    elif {"BOROUGH", "SECTOR", "YEAR", "EMPLOYEE_JOBS"}.issubset(df.columns):
        strategy = str(rng.choice(
            ["sector_vs_sector", "borough_vs_borough"],
            p=[0.60, 0.40]
        ))

        # 1) Compare two sectors across boroughs for one year
        if strategy == "sector_vs_sector":
            year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
            if not year_candidates:
                return None
            year_val = int(rng.choice(year_candidates))

            sub = df[df["YEAR"] == year_val].copy()
            top_sectors = (
                sub.groupby("SECTOR")["EMPLOYEE_JOBS"]
                .sum()
                .nlargest(10)
                .index
                .tolist()
            )
            if len(top_sectors) < 2:
                return None

            sector_a, sector_b = rng.choice(top_sectors, size=2, replace=False).tolist()

            wide = (
                sub[sub["SECTOR"].isin([sector_a, sector_b])]
                .pivot_table(index="BOROUGH", columns="SECTOR", values="EMPLOYEE_JOBS", aggfunc="sum")
                .dropna()
                .reset_index()
            )
            if len(wide) < min_points:
                return None

            plot_df = wide.rename(columns={sector_a: "x", sector_b: "y"})
            plot_df["point_id"] = plot_df["BOROUGH"].astype(str)
            plot_df["label"] = plot_df["BOROUGH"].astype(str)
            plot_df["_label"] = plot_df["BOROUGH"].astype(str)

            context = {
                "x_label": f"{sector_a.title()} Jobs",
                "y_label": f"{sector_b.title()} Jobs",
                "x_col": sector_a,
                "y_col": sector_b,
                "strategy": strategy,
                "detail": f"Borough comparison in {year_val}",
            }

        # 2) Compare two boroughs across sectors for one year
        else:
            year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
            if not year_candidates:
                return None
            year_val = int(rng.choice(year_candidates))

            sub = df[df["YEAR"] == year_val].copy()
            top_boroughs = (
                sub.groupby("BOROUGH")["EMPLOYEE_JOBS"]
                .sum()
                .nlargest(10)
                .index
                .tolist()
            )
            if len(top_boroughs) < 2:
                return None

            borough_a, borough_b = rng.choice(top_boroughs, size=2, replace=False).tolist()

            wide = (
                sub[sub["BOROUGH"].isin([borough_a, borough_b])]
                .pivot_table(index="SECTOR", columns="BOROUGH", values="EMPLOYEE_JOBS", aggfunc="sum")
                .dropna()
                .reset_index()
            )
            if len(wide) < min_points:
                return None

            plot_df = wide.rename(columns={borough_a: "x", borough_b: "y"})
            plot_df["point_id"] = wide["SECTOR"].astype(str)
            plot_df["label"] = wide["SECTOR"].astype(str)
            plot_df["_label"] = wide["SECTOR"].astype(str)

            context = {
                "x_label": f"{borough_a.title()} Jobs",
                "y_label": f"{borough_b.title()} Jobs",
                "x_col": borough_a,
                "y_col": borough_b,
                "strategy": strategy,
                "detail": f"Sector comparison in {year_val}",
            }

        plot_df = plot_df.dropna(subset=["x", "y"])
        plot_df = plot_df[np.isfinite(plot_df["x"]) & np.isfinite(plot_df["y"])]
        plot_df = plot_df[(plot_df["x"] > 0) & (plot_df["y"] > 0)]

        if len(plot_df) < min_points:
            return None

    else:
        return None

    # ------------------------------------------------------------------
    # Common group/color/shape assignment
    # ------------------------------------------------------------------
    plot_df = plot_df.reset_index(drop=True)

    n_groups = max(1, int(style.get("n_groups", 1)))
    n_groups = min(n_groups, len(plot_df))

    if n_groups > 1:
        try:
            plot_df["group"] = pd.qcut(
                plot_df["x"],
                q=n_groups,
                labels=[f"Group {i+1}" for i in range(n_groups)],
                duplicates="drop",
            ).astype(str)
        except Exception:
            plot_df["group"] = "Group 1"
    else:
        plot_df["group"] = "Group 1"

    actual_groups = plot_df["group"].unique().tolist()
    n_actual = len(actual_groups)

    n_colors = max(1, min(int(style.get("n_colors", 1)), n_actual))
    n_shapes = max(1, min(int(style.get("n_shapes", 1)), n_actual))

    color_cats = [f"Color {i+1}" for i in range(n_colors)]
    shape_cats = [f"Shape {i+1}" for i in range(n_shapes)]

    color_map = {g: color_cats[i % n_colors] for i, g in enumerate(actual_groups)}
    shape_map = {g: shape_cats[i % n_shapes] for i, g in enumerate(actual_groups)}

    plot_df["color_group"] = plot_df["group"].map(color_map)
    plot_df["shape_group"] = plot_df["group"].map(shape_map)
    plot_df["show_label"] = False

    label_code = int(style.get("direct_labels", 0))
    if label_code == 1:
        plot_df["show_label"] = True
    elif label_code == 2:
        n = min(max(3, int(round(len(plot_df) * 0.30))), len(plot_df))
        idx = rng.choice(plot_df.index.to_numpy(), size=n, replace=False)
        plot_df.loc[idx, "show_label"] = True

    context.update({
        "n_points": int(len(plot_df)),
        "n_groups": int(plot_df["group"].nunique()),
        "orientation": int(style.get("scatter_orientation", 1)),
        "x_range": [float(plot_df["x"].min()), float(plot_df["x"].max())],
        "y_range": [float(plot_df["y"].min()), float(plot_df["y"].max())],
    })

    return plot_df, context

In [125]:

def n_points_from_bin(rng: np.random.Generator, code: int) -> int:
    # codes: 0=0-10, 1=10-20, 2=20-30, 3=30+, 6=edge case -> 30+
    mapping = {0: (6, 11), 1: (10, 21), 2: (20, 31), 3: (30, 61), 6: (30, 61)}
    lo, hi = mapping.get(code, (10, 21))
    return int(rng.integers(lo, hi))


def scale_bounds(code: int, axis: str = "x") -> tuple[float, float]:
    table = X_SCALE_BOUNDS if axis == "x" else Y_SCALE_BOUNDS
    return table.get(code, (0, 100))


def build_palette(
    n: int,
    style: dict,
    rng: np.random.Generator | None = None,
) -> list[str]:
    """
    Scatter-compatible wrapper around the line-generator palette logic.
    Uses contrast-aware, randomly jittered palettes.
    """
    return color_list(style, max(1, n), rng=rng)


def build_shape_sequence(style: dict, n_shapes: int) -> list[str]:
    mode = style.get("point_shape_mode", 0)
    if mode == 1:   # squares
        return ["square"]
    if mode in {2, 4}:  # by_group
        base = ["circle", "square", "diamond", "triangle-up", "cross"]
        return base[:max(1, n_shapes)]
    if mode == 7:   # triangles
        return ["triangle-up"]
    if mode == 8:   # crosses
        return ["cross"]
    return ["circle"]   # dots (0), dots_outline (3), dots_overlap (5), etc.

# UNCOMMENT FOR GROQ
#_groq_client = Groq()  # reads GROQ_API_KEY from env

MAX_TITLE_CHARS = 60
def make_title(context: dict, style: dict | None = None) -> tuple[str, str | None]:
    x_label = context.get("x_label", "X")
    y_label = context.get("y_label", "Y")
    n_groups = context.get("n_groups", 1)
    orientation = context.get("orientation", 1)

    orientation_desc = {
        0: "negative correlation", 1: "positive correlation",
        2: "no clear trend", 3: "clustered groups",
        4: "horizontal bands", 5: "vertical bands", 6: "parabolic relationship",
    }.get(int(orientation), "relationship")

    need_subtitle = style is not None and int(style.get("subtitle_present", 0)) == 1

    system_prompt = (
        "You generate short, realistic chart titles for scatter plots — the kind you'd see "
        "in a business report, scientific paper, or dashboard. "
        "Keep titles under 60 characters. Keep subtitles under 60 characters. "
        "Be concise and natural. No quotes, no markdown."
    )
    user_prompt = (
        f"Generate a chart title for a scatter plot showing the {orientation_desc} "
        f"between {x_label} and {y_label}"
        + (f" across {n_groups} groups" if n_groups > 1 else "")
        + ".\n"
    )
    if need_subtitle:
        user_prompt += (
            "Also generate a short subtitle (one line, adds context or detail). "
            "Respond in this exact format:\n"
            "TITLE: <title here>\n"
            "SUBTITLE: <subtitle here>"
        )
    else:
        user_prompt += "Respond with just the title, nothing else."

    try:
        response = _groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.8,
            max_tokens=80,
        )
        raw = response.choices[0].message.content.strip()

        if need_subtitle and "TITLE:" in raw:
            lines = {k.strip(): v.strip() for k, v in
                     (line.split(":", 1) for line in raw.splitlines() if ":" in line)}
            return lines.get("TITLE", raw)[:MAX_TITLE_CHARS], lines.get("SUBTITLE", "")[:MAX_TITLE_CHARS]

        return raw[:MAX_TITLE_CHARS], None

    except Exception:
        # Fallback when Groq is unavailable or rate limited
        fallback_title = f"{y_label} vs {x_label}"[:MAX_TITLE_CHARS]
        fallback_sub = f"{orientation_desc.title()}" if need_subtitle else None
        return fallback_title, fallback_sub


def make_subtitle(context: dict) -> str:
    return f"{context['n_points']} points · {context['n_groups']} group(s) · pattern={context['orientation']}"


def choose_label_indices(df: pd.DataFrame, style: dict, rng: np.random.Generator) -> np.ndarray:
    code = style.get("direct_labels", 0)
    if code == 0:   # none
        return np.array([], dtype=int)
    if code == 1:   # all
        return df.index.to_numpy()
    # code 2: partial
    n = min(max(3, int(round(len(df) * 0.30))), len(df))
    return np.sort(rng.choice(df.index.to_numpy(), size=n, replace=False))


def compute_regression_df(df: pd.DataFrame, style: dict, x_col: str, y_col: str) -> pd.DataFrame:
    empty = pd.DataFrame(columns=[x_col, "yhat", "y_upper", "y_lower", "line_id", "group"])
    reg_code = style.get("regression_line", 0)
    if reg_code == 0:
        return empty

    if reg_code == 1:
        groups = [("overall", df.copy())]
    else:
        group_names = list(df["group"].dropna().unique())
        if len(group_names) >= 2:
            groups = [(str(g), df[df["group"] == g].copy()) for g in group_names[:2]]
        else:
            split = df[x_col] <= df[x_col].median()
            groups = [("segment_1", df[split].copy()), ("segment_2", df[~split].copy())]

    outputs = []
    for line_idx, (group_name, gdf) in enumerate(groups, start=1):
        if len(gdf) < 2:
            continue
        x = gdf[x_col].to_numpy(dtype=float)
        y = gdf[y_col].to_numpy(dtype=float)
        slope, intercept = np.polyfit(x, y, 1)
        xs = np.linspace(float(x.min()), float(x.max()), 60)
        yhat = intercept + slope * xs
        resid = y - (intercept + slope * x)
        sigma = float(np.std(resid)) if len(resid) > 1 else 0.0
        show_band = style.get("regression_confidence_band", 0) == 1
        band = 0.0 if not show_band else max(sigma, 0.02 * max(1.0, float(y.max())))
        outputs.append(pd.DataFrame({
            x_col: xs, "yhat": yhat,
            "y_upper": yhat + band, "y_lower": yhat - band,
            "line_id": f"line_{line_idx}", "group": group_name,
        }))

    return pd.concat(outputs, ignore_index=True) if outputs else empty


def sample_scatter_data(rng: np.random.Generator, style: dict) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    n_points = n_points_from_bin(rng, int(style.get("n_points_bin", 3)))
    n_groups = max(1, int(style.get("n_groups", 1)))

    x_min, x_max = scale_bounds(style.get("x_scale", 0), "x")
    y_min, y_max = scale_bounds(style.get("y_scale", 0), "y")

    # Fallback if bounds are degenerate
    if x_max <= x_min: x_min, x_max = 0.0, 20.0
    if y_max <= y_min: y_min, y_max = 0.0, 100.0

    groups = [f"Group {i+1}" for i in range(n_groups)]
    point_ids = [f"P{i+1}" for i in range(n_points)]
    group_assign = np.array([groups[i % n_groups] for i in range(n_points)])
    rng.shuffle(group_assign)

    x = np.zeros(n_points)
    y = np.zeros(n_points)
    orientation = style.get("scatter_orientation", 1)

    if orientation == 3:    # groups
        x_centers = np.linspace(x_min + 0.15*(x_max-x_min), x_max - 0.15*(x_max-x_min), n_groups)
        y_centers = np.linspace(y_min + 0.25*(y_max-y_min), y_max - 0.25*(y_max-y_min), n_groups)
        rng.shuffle(y_centers)
        for idx, g in enumerate(groups):
            mask = group_assign == g
            x[mask] = rng.normal(x_centers[idx], 0.06*(x_max-x_min), mask.sum())
            y[mask] = rng.normal(y_centers[idx], 0.08*(y_max-y_min), mask.sum())
    elif orientation == 4:  # horizontal lines
        for idx, g in enumerate(groups):
            mask = group_assign == g
            y_center = y_min + (idx+1)/(n_groups+1) * (y_max-y_min)
            x[mask] = rng.uniform(x_min, x_max, mask.sum())
            y[mask] = rng.normal(y_center, 0.02*(y_max-y_min), mask.sum())
    elif orientation == 5:  # vertical lines
        for idx, g in enumerate(groups):
            mask = group_assign == g
            x_center = x_min + (idx+1)/(n_groups+1) * (x_max-x_min)
            x[mask] = rng.normal(x_center, 0.02*(x_max-x_min), mask.sum())
            y[mask] = rng.uniform(y_min, y_max, mask.sum())
    elif orientation == 6:  # parabola
        x = rng.uniform(x_min, x_max, n_points)
        x_norm = (x - x_min) / max(1e-9, x_max - x_min)
        y_norm = 4 * x_norm * (1 - x_norm) + rng.normal(0, 0.05, n_points)
        y = y_min + np.clip(y_norm, 0, 1) * (y_max - y_min)
    else:
        x = rng.uniform(x_min, x_max, n_points)
        x_norm = (x - x_min) / max(1e-9, x_max - x_min)
        noise = rng.normal(0, 0.10, n_points)
        if orientation == 1:    # ascending
            y_norm = 0.10 + 0.78 * x_norm + noise
        elif orientation == 0:  # descending
            y_norm = 0.88 - 0.75 * x_norm + noise
        else:                   # blob (2)
            y_norm = 0.50 + rng.normal(0, 0.18, n_points)
        y = y_min + np.clip(y_norm, 0.02, 0.98) * (y_max - y_min)
        if n_groups > 1:
            offsets = np.linspace(-0.10, 0.10, n_groups)
            for idx, g in enumerate(groups):
                mask = group_assign == g
                y[mask] += offsets[idx] * (y_max - y_min)

    x = np.clip(x, x_min, x_max)
    y = np.clip(y, y_min, y_max)

    n_colors = int(style.get("n_colors", 1))
    n_shapes = int(style.get("n_shapes", 1))
    color_count = max(1, min(n_colors, n_groups if n_groups > 1 else n_colors))
    shape_count = max(1, min(n_shapes, n_groups if n_groups > 1 else n_shapes))

    color_categories = [f"Color {i+1}" for i in range(color_count)]
    shape_categories = [f"Shape {i+1}" for i in range(shape_count)]
    color_group_map = {g: color_categories[i % color_count] for i, g in enumerate(groups)}
    shape_group_map = {g: shape_categories[i % shape_count] for i, g in enumerate(groups)}

    df = pd.DataFrame({
        "point_id": point_ids, "x": x, "y": y,
        "group": group_assign,
        "color_group": [color_group_map[g] for g in group_assign],
        "shape_group": [shape_group_map[g] for g in group_assign],
        "label": point_ids,
    }).sort_values(["group", "x", "y"]).reset_index(drop=True)

    label_idx = choose_label_indices(df, style, rng)
    df["show_label"] = False
    df.loc[label_idx, "show_label"] = True

    reg_df = compute_regression_df(df, style, x_col="x", y_col="y")

    context = {
        "x_label": "X value", "y_label": "Y value",
        "n_points": int(n_points), "n_groups": int(n_groups),
        "orientation": orientation, "group_mode": "group",
        "x_range": [float(x_min), float(x_max)],
        "y_range": [float(y_min), float(y_max)],
    }
    return df, reg_df, context


## 3. Shared styling helpers


In [126]:
# ── Section 3: Shared styling helpers ───────────────────────────────────────
# WCAG contrast-aware colour helpers — same logic as the line generator.

# ── Colour lookup tables (numeric code → value) ───────────────────────────────

BACKGROUNDS = {
    0: "transparent",
    1: "white",
    2: "light_gray",
    3: "dark_gray",
    4: "light_color",
}
BACKGROUND_COLOR_BASES = {
    0: None,                  # transparent
    1: (255, 255, 255),       # white
    2: (242, 242, 242),       # light gray
    3: (77,  77,  77),        # dark gray
    4: (237, 244, 255),       # light blue
    5: (235, 235, 235),       # outer gray
    6: (30,  30,  50),        # dark
}

TITLE_COLOR_BASES = {
    0: (0, 0, 0),
    1: (255, 127, 0),
    2: (200, 30, 30),
    3: (100, 100, 100),
    4: (0, 0, 0),
    5: (30, 150, 30),
    6: (255, 255, 255),
}
TITLE_SIZE_BASES = {0: 16, 1: 12, 2: 20, 3: 16}

TITLE_LOCATIONS = {
    0: ("none",   0.50, "center", "middle"),
    1: ("center", 0.50, "center", "middle"),
    2: ("left",   0.01, "left",   "start"),
    3: ("right",  0.99, "right",  "end"),
}

LEGEND_ORIENTATIONS = {
    0: None,
    1: ("left",      "center left",  (-0.22, 0.5),  dict(x=-0.22, y=0.5,  xanchor="right",  yanchor="middle")),
    2: ("right",     "center left",  (1.02,  0.5),  dict(x=1.02,  y=0.5,  xanchor="left",   yanchor="middle")),
    3: ("top",       "lower center", (0.5,   1.04), dict(x=0.5,   y=1.02, xanchor="center", yanchor="bottom")),
    4: ("bottom", "upper center", (0.5, -0.12), dict(x=0.5, y=-0.22, xanchor="center", yanchor="top")),
    5: ("top-left",  "lower left",   (0.0,   1.02), dict(x=0.0,   y=1.02, xanchor="left",   yanchor="bottom")),
    6: ("top-right", "lower right",  (1.0,   1.02), dict(x=1.0,   y=1.02, xanchor="right",  yanchor="bottom")),
    7: ("top-right", "lower right",  (1.0,   1.02), dict(x=1.0,   y=1.02, xanchor="right",  yanchor="bottom")),
    8: ("top",       "lower center", (0.5,   1.04), dict(x=0.5,   y=1.02, xanchor="center", yanchor="bottom")),
}

# Palette base colors per code — each is a list of RGB tuples
# Variation is applied per color via jitter at render time
PALETTE_BASES = {
    0:  [(0,   0,   0  )] * 6,                                          # black
    1:  [(30,  30,  30 ), (60,  60,  60 ), (90,  90,  90 ),
         (40,  50,  60 ), (50,  40,  70 ), (40,  70,  50 )],            # dark
    2:  [(31,  119, 180), (255, 127, 14 ), (44,  160, 44 ),
         (214, 39,  40 ), (148, 103, 189), (23,  190, 207)],            # bright
    3:  [(255, 140, 0  ), (230, 100, 0  ), (200, 80,  0  ),
         (255, 165, 50 ), (210, 120, 20 ), (240, 150, 30 )],            # orange
    4:  [(0,   30,  100), (0,   50,  130), (0,   70,  160),
         (20,  60,  140), (10,  40,  120), (30,  80,  150)],            # dark blue
    5:  [(50,  50,  50 ), (100, 100, 100), (150, 150, 150),
         (180, 180, 180), (200, 200, 200), (80,  80,  80 )],            # grey shades
    6:  [(0,   100, 200), (30,  130, 220), (60,  160, 240),
         (0,   80,  180), (20,  110, 210), (50,  140, 230)],            # bright blue
    7:  [(180, 220, 255), (150, 200, 240), (200, 230, 255),
         (160, 210, 245), (170, 215, 250), (140, 195, 235)],            # light multicolor
    8:  [(0,   0,   0  ), (0,   80,  160), (180, 30,  30 ),
         (0,   40,  120), (140, 20,  20 ), (20,  60,  140)],            # black, blue, red
    9:  [(20,  20,  60 ), (60,  20,  80 ), (20,  60,  40 ),
         (40,  40,  80 ), (80,  20,  60 ), (20,  80,  60 )],            # dark multicolor
    10: [(180, 0,   0  ), (0,   150, 0  ), (0,   0,   200),
         (160, 0,   0  ), (0,   130, 0  ), (0,   0,   180)],            # RGB
    11: [(180, 30,  30 ), (30,  80,  180), (180, 30,  30 ),
         (30,  80,  180), (160, 20,  20 ), (20,  60,  160)],            # red, blue
    12: [(100, 100, 120), (120, 100, 110), (110, 120, 100),
         (90,  110, 120), (115, 105, 95 ), (105, 115, 110)],            # muted multicolor
    13: [(0,   80,  160), (180, 80,  0  ), (200, 30,  30 ),
         (0,   60,  140), (160, 60,  0  ), (180, 20,  20 )],            # blue, red, orange
    14: [(0,   60,  160), (30,  90,  180), (60,  120, 200),
         (10,  70,  170), (40,  100, 190), (20,  80,  175)],            # blue shades
    15: [(180, 30,  30 ), (220, 100, 30 ), (60,  160, 60 ),
         (160, 20,  20 ), (200, 80,  20 ), (40,  140, 40 )],            # red, orange, green
    16: [(255, 255, 255)] * 6,                                          # white (dark bg)
    17: [(30,  160, 80 ), (0,   100, 180), (60,  180, 100),
         (20,  140, 60 ), (0,   80,  160), (40,  160, 90 )],            # green, blue
    18: [(31,  119, 180), (255, 127, 14 ), (44,  160, 44 ),
         (214, 39,  40 ), (148, 103, 189), (23,  190, 207)],            # fallback bright
}

AXIS_TEXT_COLORS = {0: "black", 1: None, 2: "green", 3: None, 4: "lightgray"}
AXIS_COLOR_BASES = {
    0: (0,   0,   0),    # black
    1: (80,  80,  80),   # dark gray
    2: (200, 200, 200),  # light gray
    4: (255, 255, 255),  # white (dark background)
}
GRIDLINE_COLOR_BASES = {
    1: (140, 140, 140),   # grey
    2: (0,   0,   0),     # black
    3: (200, 200, 200),   # light grey
    4: (31,  119, 180),   # blue
    5: (140, 140, 140),   # grey (dashed)
    6: (31,  119, 180),   # blue for dark bg
    7: (255, 255, 255),   # white for color bg
    8: (31,  119, 180),   # blue for light bg
}
LABEL_COLORS     = {0: "black", 1: None, 2: "black", 3: "bright"}

LINE_STRUCTURES   = {0: "straight", 1: "smooth"}
LINE_PATTERNS_MPL = {0: "-", 1: ":", 2: "--"}
LINE_PATTERNS_PLY = {0: "solid", 1: "dot", 2: "dash"}

POINT_SHAPE_MODES = {0: "dots", 1: "squares", 2: "by_line", 3: "none"}
POINT_SAME_COLORS = {0: "different", 1: "same", 2: "shade", 3: "none"}

# BRIGHT_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#17becf"]
# DARK_COLORS   = ["#111111", "#3a3a3a", "#555555", "#23415a", "#4a2c5f", "#31533b"]
# BLACK_COLORS  = ["#000000"] * 6
# PALETTE_TYPES = {0: BLACK_COLORS, 1: DARK_COLORS, 2: BRIGHT_COLORS}

BRIGHT_COLORS = [f"#{r:02x}{g:02x}{b:02x}" for r, g, b in PALETTE_BASES[2]]

MPL_MARKERS = {"dots": "o", "squares": "s", "by_line": "o", "none": None}
ALT_SHAPES  = {"dots": "circle", "squares": "square", "by_line": "circle"}
PLY_MARKERS = ["circle", "square", "triangle-up", "diamond", "cross", "x"]

X_MAJOR_TICK_VALUES = {0: "auto", 1: 1, 2: 2, 3: 5, 4: 10, 5: 25, 6: 50}
X_MINOR_TICK_VALUES = {0: 0, 1: 1, 2: 4, 3: 9}
Y_MAJOR_TICK_VALUES = {0: "auto", 1: 1, 2: 2, 3: 5, 4: 10, 5: 25, 6: 50, 7: 100, 8: 200}
Y_MINOR_TICK_VALUES = {0: 0, 1: 1, 2: 4, 3: 9}

# ── Helper functions ──────────────────────────────────────────────────────────

def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    hex_color = hex_color.lstrip("#")
    return int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)


def _relative_luminance(hex_color: str) -> float:
    """WCAG relative luminance of a hex color, in [0, 1]."""
    r, g, b = (_hex_to_rgb(hex_color))
    def linearize(c):
        c /= 255
        return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * linearize(r) + 0.7152 * linearize(g) + 0.0722 * linearize(b)


def _contrast_ratio(hex_a: str, hex_b: str) -> float:
    """WCAG contrast ratio between two hex colors."""
    la = _relative_luminance(hex_a)
    lb = _relative_luminance(hex_b)
    lighter, darker = max(la, lb), min(la, lb)
    return (lighter + 0.05) / (darker + 0.05)

def _sample_with_contrast(base: tuple[int, int, int], bg: str,
                           rng: np.random.Generator, v: int = 30) -> str:
    check_contrast = bg not in {"none", "transparent"}
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate
    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def get_background_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("background", 1))
    base = BACKGROUND_COLOR_BASES.get(code)
    if base is None:
        return "none"
    rng = rng or np.random.default_rng()
    v   = 8  # small jitter — backgrounds should be subtle
    r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
    g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
    b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
    return f"#{r:02x}{g:02x}{b:02x}"


def get_foreground_color(style: dict) -> str:
    return "white" if int(style.get("background", 1)) in {3, 6} else "black"


def get_title_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("title_color", 0))
    base = TITLE_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    # Skip contrast check for transparent background
    check_contrast = bg not in {"none", "transparent"}

    v = 30
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    # Fallback: return pure black or white depending on background luminance
    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def get_axis_color(style: dict, rng: np.random.Generator | None = None) -> str | None:
    """Returns the axis line color, or None for code 3 (no axes)."""
    code = int(style.get("axis_color", 0))
    if code == 3:
        return None
    base = AXIS_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)
    return _sample_with_contrast(base, bg, rng, v=20)

def get_legend_title_fontsize(style: dict, rng: np.random.Generator | None = None) -> int | None:
    if int(style.get("legend_title_size", 0)) != 1:
        return None
    rng = rng or np.random.default_rng()
    return int(np.clip(11 + rng.integers(-1, 2), 8, 16))


def get_title_fontsize(style: dict, rng: np.random.Generator | None = None) -> int:
    code = int(style.get("title_size", 0))
    base = TITLE_SIZE_BASES.get(code, 16)
    rng  = rng or np.random.default_rng()
    return int(np.clip(base + rng.integers(-2, 3), 8, 32))


def get_title_location(style: dict):
    """Returns (label, x_pos, halign, anchor)."""
    return TITLE_LOCATIONS.get(int(style.get("title_location", 1)),
                               ("center", 0.50, "center", "middle"))


def get_axis_text_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("axis_text_color", 0))
    if code == 1:
        return get_title_color(style, rng=rng)
    if code == 3:
        bg = get_background_color(style, rng=rng)
        return bg if bg not in {"none", "transparent"} else "white"
    return AXIS_TEXT_COLORS.get(code, "black") or "black"


def get_gridline_style(style: dict, rng: np.random.Generator | None = None) -> tuple[str, str]:
    """Returns (color, linestyle) where linestyle is 'solid' or 'dashed'."""
    code = int(style.get("gridline_color", 1))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    base = GRIDLINE_COLOR_BASES.get(code, (140, 140, 140))
    linestyle = "dashed" if code == 5 else "solid"

    color = _sample_with_contrast(base, bg, rng, v=15)
    return color, linestyle


def get_label_color(style: dict, rng: np.random.Generator | None = None,
                    line_color: str | None = None) -> str:
    code = int(style.get("label_color", 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    if code == 1:
        return get_title_color(style, rng=rng)

    if code == 2:
        return _sample_with_contrast((0, 0, 0), bg, rng, v=20)

    if code == 3:
        return line_color or BRIGHT_COLORS[0]

    if code == 4:
        base = 140
        v    = 40
        gray = int(np.clip(base + rng.integers(-v, v + 1), 80, 220))
        hex_gray = f"#{gray:02x}{gray:02x}{gray:02x}"
        # Retry if contrast is poor
        bg_check = bg not in {"none", "transparent"}
        for _ in range(20):
            gray = int(np.clip(base + rng.integers(-v, v + 1), 80, 220))
            hex_gray = f"#{gray:02x}{gray:02x}{gray:02x}"
            if not bg_check or _contrast_ratio(hex_gray, bg) >= 3.0:
                return hex_gray
        return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"

    # code 0 or fallback — black with contrast check
    return _sample_with_contrast((0, 0, 0), bg, rng, v=10)

def get_legend_fill_and_outline(style: dict, rng: np.random.Generator | None = None) -> tuple[str | None, bool]:
    """Returns (fill_color, has_outline). Encodes all legend_fill codes in one place."""
    rng  = rng or np.random.default_rng()
    code = int(style.get("legend_fill", 0))

    if code == 0:
        return None, False

    if code == 1:
        # Gray fill, no outline — sample a gray shade for variation
        base = 180
        v    = 30
        gray = int(np.clip(base + rng.integers(-v, v + 1), 120, 235))
        return f"#{gray:02x}{gray:02x}{gray:02x}", False

    if code == 2:
        # No fill, no outline
        return None, False

    if code in {3, 4}:
        # White fill with black outline
        return "#ffffff", True

    return None, False


def get_legend_text_color(style: dict, rng: np.random.Generator | None = None) -> str | None:
    code = int(style.get("legend_text_color", 0))
    if code == 1:
        return get_title_color(style, rng=rng)
    if code == 2:
        return None

    base_color = {0: "#000000", 3: "#555555", 4: "#ffffff", 5: "#888888", 6: "#333333"}.get(code, "#000000")

    bg = get_background_color(style)
    if bg in {"none", "transparent"}:
        return base_color

    r, g, b = _hex_to_rgb(base_color)
    rng = rng or np.random.default_rng()
    v = 20
    for _ in range(20):
        rc = int(np.clip(r + rng.integers(-v, v + 1), 0, 255))
        gc = int(np.clip(g + rng.integers(-v, v + 1), 0, 255))
        bc = int(np.clip(b + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{rc:02x}{gc:02x}{bc:02x}"
        if _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"

LEGEND_TITLE_COLOR_BASES = {
    0: (0, 0, 0),        # black
    1: (100, 100, 100),  # gray
    2: (255, 255, 255),  # white
}

def get_legend_title_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("legend_title_color", 0))
    base = LEGEND_TITLE_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    check_contrast = bg not in {"none", "transparent"}

    v = 20
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def color_list(style: dict, n: int, rng: np.random.Generator | None = None) -> list:
    code  = int(style.get("palette_type", 2))
    rng   = rng or np.random.default_rng()
    bases = PALETTE_BASES.get(code, PALETTE_BASES[2])
    bg    = get_background_color(style)
    v     = 15

    def _too_similar(candidate: str, existing: list[str], threshold: int = 60) -> bool:
        cr, cg, cb = _hex_to_rgb(candidate)
        for ex in existing:
            er, eg, eb = _hex_to_rgb(ex)
            if abs(cr - er) + abs(cg - eg) + abs(cb - eb) < threshold:
                return True
        return False

    colors = []
    for i in range(n):
        base = bases[i % len(bases)]
        chosen = None
        for _ in range(40):
            r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
            g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
            b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
            candidate = f"#{r:02x}{g:02x}{b:02x}"
            bg_ok      = bg in {"none", "transparent"} or _contrast_ratio(candidate, bg) >= 2.0
            similar_ok = not _too_similar(candidate, colors)
            if bg_ok and similar_ok:
                chosen = candidate
                break
        if chosen is None:
            # Fallback — spread evenly through hue space
            chosen = "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"
        colors.append(chosen)

    return colors


def line_pattern_for(style: dict, idx: int, renderer: str = "mpl") -> str:
    code = int(style.get("line_pattern", 0))
    if code == 3:   # mixed
        sub = idx % 3
        return LINE_PATTERNS_MPL.get(sub, "-") if renderer == "mpl" else LINE_PATTERNS_PLY.get(sub, "solid")
    return LINE_PATTERNS_MPL.get(code, "-") if renderer == "mpl" else LINE_PATTERNS_PLY.get(code, "solid")


def marker_for(style: dict, idx: int):
    mode = POINT_SHAPE_MODES.get(int(style.get("point_shape_mode", 0)), "dots")
    if mode == "none":
        return None
    if mode == "by_line":
        return ["o", "s", "^", "D", "P", "X"][idx % 6]
    return MPL_MARKERS.get(mode, "o")


def point_color(line_color: str, style: dict, idx: int) -> str:
    mode = POINT_SAME_COLORS.get(int(style.get("point_same_color", 1)), "same")
    if mode == "none":
        return line_color
    if mode == "different":
        return BRIGHT_COLORS[(idx + 2) % len(BRIGHT_COLORS)]
    if mode == "shade":
        return "#999999"
    return line_color


def legend_location_mpl(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    if not entry:
        return None, None
    _, loc, bbox, _ = entry
    return loc, bbox

def legend_orient_altair(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[0] if entry else None

def legend_coords_plotly(style: dict) -> dict:
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[3] if entry else {}

def legend_orient_altair(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[0] if entry else None


def apply_gridlines_mpl(ax, style: dict, rng: np.random.Generator | None = None):
    grid = int(style.get("gridlines", 0))
    if grid == 0:
        ax.grid(False)
        return

    color, linestyle = get_gridline_style(style, rng=rng)
    mpl_linestyle = "--" if linestyle == "dashed" else "-"

    if grid == 2:
        ax.yaxis.grid(True, color=color, linewidth=0.6, linestyle=mpl_linestyle, alpha=0.75, zorder=0)
        ax.xaxis.grid(False)
    elif grid in {1, 3, 4}:
        width = 0.4 if grid == 3 else 0.8 if grid == 4 else 0.6
        ax.grid(True, color=color, linewidth=width, linestyle=mpl_linestyle, alpha=0.75, zorder=0)
    else:
        ax.grid(False)


def apply_outline_mpl(ax, fig, style: dict, rng: np.random.Generator | None = None):
    outline  = int(style.get("chart_outline", 1))
    fg       = get_foreground_color(style)
    axis_c   = get_axis_color(style, rng=rng)

    if axis_c is None:
        # code 3 — no axes at all
        for s in ax.spines.values():
            s.set_visible(False)
        ax.tick_params(left=False, bottom=False)
        if int(style.get("image_outline", 0)) == 1:
            fig.patch.set_edgecolor(fg)
            fig.patch.set_linewidth(1.2)
        return

    for spine in ax.spines.values():
        spine.set_color(axis_c)

    if outline == 0:
        for s in ax.spines.values():
            s.set_visible(False)
    elif outline == 1:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    elif outline == 2:
        pass  # full box
    elif outline == 4:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.yaxis.set_visible(False)

    if int(style.get("image_outline", 0)) == 1:
        fig.patch.set_edgecolor(fg)
        fig.patch.set_linewidth(1.2)

# ── Scatter-specific helpers ──────────────────────────────────────────────────

GRAY_FILLS = ["#e8e8e8", "#dedede", "#d8d8d8", "#e2e2e2", "#dcdcdc", "#e0e0e0", "#d4d4d4"]

def get_legend_fill_color(style: dict, rng: np.random.Generator | None = None) -> str | None:
    legend_fill = int(style.get("legend_fill", 0))
    if legend_fill == 1:
        rng = rng or np.random.default_rng()
        return GRAY_FILLS[int(rng.integers(0, len(GRAY_FILLS)))]
    if legend_fill == 3:
        return "white"
    return None

X_TICK_STEPS = {
    0: 5, 1: 2, 2: 0.1, 3: 10, 4: 500, 5: 0.5, 6: 0.2,
    7: 5, 8: None, 9: 20, 10: 100, 11: 50000, 12: 5000,
    13: 1, 14: 0.25, 15: 25, 16: 50, 17: 1000, 18: None,
    19: 1000000, 20: 30, 21: 200, 22: 20000, 23: 2, 24: 0.1, 25: 20,
}

Y_TICK_STEPS = {
    0: 2, 1: 5, 2: 100, 3: 0.5, 4: 1000, 5: 1, 6: 0.25,
    7: 20, 8: 10, 9: None, 10: 25, 11: 0.2, 12: 50000,
    13: 50, 14: 500, 15: 1000000, 16: 8, 17: 2, 18: 2000,
    19: 5000, 20: 20000, 21: 10000, 22: 0.1,
}

def get_tick_step(style: dict, axis: str = "x") -> float:
    code = int(style.get(f"{axis}_tick_step", 0))
    table = X_TICK_STEPS if axis == "x" else Y_TICK_STEPS
    step = table.get(code)
    lo, hi = scale_bounds(int(style.get(f"{axis}_scale", 0)), axis)
    span = hi - lo
    if not step or step <= 0 or (span / step) > 50:
        step = span / 10
    return max(step, 1e-6)

def get_mpl_marker(shape_name: str) -> str:
    return {"circle": "o", "square": "s", "diamond": "D",
            "triangle-up": "^", "x": "x", "cross": "+"}.get(shape_name, "o")

def get_plotly_symbol(shape_name: str) -> str:
    return {"circle": "circle", "square": "square", "diamond": "diamond",
            "triangle-up": "triangle-up", "x": "x", "cross": "cross"}.get(shape_name, "circle")

def get_altair_shape(shape_name: str) -> str:
    return {"circle": "circle", "square": "square", "diamond": "diamond",
            "triangle-up": "triangle-up", "x": "cross",
            "cross": "diamond-cross"}.get(shape_name, "circle")

def legend_position_altair(style: dict) -> tuple[str | None, str | None]:
    code = int(style.get("legend_orientation", 2))
    entry = LEGEND_ORIENTATIONS.get(code)
    if not entry:
        return None, None
    orient = entry[0]
    return orient, "middle" if orient in {"left", "right"} else "top"

def apply_gridline_state_mpl(ax, style: dict, rng=None):
    grid = int(style.get("gridlines", 0))
    color, linestyle = get_gridline_style(style)
    ls = "--" if linestyle == "dashed" else "-"
    if grid == 0:
        ax.grid(False)
    elif grid == 2:
        ax.yaxis.grid(True, color=color, linewidth=0.8, alpha=0.7, linestyle=ls, zorder=0)
        ax.xaxis.grid(False)
    elif grid == 1:
        ax.grid(True, color=color, linewidth=0.8, alpha=0.7, linestyle=ls, zorder=0)
    else:
        ax.grid(True, color=color, linewidth=0.9, alpha=0.8)
        ax.minorticks_on()
        ax.grid(True, which="minor", color=color, linewidth=0.45, alpha=0.35)
    if int(style.get("minor_ticks_x", 0)) == 1 and grid == 0:
        ax.xaxis.set_minor_locator(plt.AutoMinorLocator())
    if int(style.get("minor_ticks_y", 0)) == 1 and grid == 0:
        ax.yaxis.set_minor_locator(plt.AutoMinorLocator())


def get_gridline_color(style: dict, rng: np.random.Generator | None = None) -> str:
    """Shim for renderers that call get_gridline_color — returns just the color string."""
    color, _ = get_gridline_style(style, rng=rng)
    return color


def get_altair_title_anchor(style: dict) -> str:
    _, _, _, anchor = TITLE_LOCATIONS.get(int(style.get("title_location", 1)), (None, 0.5, "center", "middle"))
    return anchor

def get_title_alignment(style: dict) -> tuple[float, str]:
    _, x, halign, _ = TITLE_LOCATIONS.get(int(style.get("title_location", 1)), (None, 0.5, "center", "middle"))
    return x, halign

def get_plotly_title_anchor(style: dict) -> tuple[float, str]:
    _, x, halign, _ = TITLE_LOCATIONS.get(int(style.get("title_location", 1)), (None, 0.5, "center", "middle"))
    return x, halign


## Harmonize styles


In [127]:
def harmonize_style(style: dict) -> dict:
    style = dict(style)

    # Define background context first — used by multiple color checks below
    bg = int(style.get("background", 1))
    dark_backgrounds = {3, 6}

    # Title consistency
    if int(style.get("title_present", 0)) == 0:
        style["title_location"] = 0
        style["subtitle_present"] = 0
        style["title_size"] = 0
        style["title_color"] = 0

    # Legend consistency
    if int(style.get("legend_present", 0)) == 0 or int(style.get("legend_orientation", 0)) == 0:
        style["legend_present"] = 0
        style["legend_orientation"] = 0
        style["legend_title_size"] = 0
        style["legend_title_color"] = 0
        style["legend_text_color"] = 2
        style["legend_outline"] = 0
        style["legend_fill"] = 0

    # Label consistency
    if int(style.get("direct_labels", 0)) == 0 or int(style.get("label_content", 0)) == 0:
        style["direct_labels"] = 0
        style["label_content"] = 0

    # Gridline consistency
    if int(style.get("gridlines", 0)) == 0:
        style["gridline_color"] = 0
        style["minor_ticks_x"] = 0
        style["minor_ticks_y"] = 0

    # Regression consistency
    if int(style.get("regression_line", 0)) == 0:
        style["regression_line_format"] = 0
        style["regression_line_color"] = 0
        style["regression_confidence_band"] = 0
        style["regression_confidence_bounds"] = 0

    # Shape consistency
    if int(style.get("point_shape_mode", 0)) not in {2, 4}:
        style["n_shapes"] = 1

    # Coerce ints
    style["n_groups"] = int(style.get("n_groups", 1))
    style["n_shapes"] = int(style.get("n_shapes", 1))
    style["n_colors"] = int(style.get("n_colors", 1))

    # Title color must be readable against background
    title_color = int(style.get("title_color", 0))
    if bg in dark_backgrounds and title_color not in {6}:
        style["title_color"] = 6    # force white
    elif bg not in dark_backgrounds and title_color in {6}:
        style["title_color"] = 0    # force black

    # Axis text color must be readable against background
    axis_text = int(style.get("axis_text_color", 0))
    if bg in dark_backgrounds and axis_text == 0:
        style["axis_text_color"] = 4    # force white on dark background
    elif bg not in dark_backgrounds and axis_text == 4:
        style["axis_text_color"] = 0    # force black on light background

    # Legend text color must be readable against background
    if int(style.get("legend_present", 0)) == 1:
        legend_text = int(style.get("legend_text_color", 0))
        if bg in dark_backgrounds and legend_text in {0, 3}:
            style["legend_text_color"] = 4    # force white
        elif bg not in dark_backgrounds and legend_text in {4}:
            style["legend_text_color"] = 0    # force black

    return style

## 4. Scatter renderers


In [128]:
def render_scatter_altair(plot_df: pd.DataFrame, reg_df, title: str, subtitle: str | None, style: dict, rng: np.random.Generator | None = None, x_label: str = "X value", y_label: str = "Y value") -> alt.Chart:
    bg = get_background_color(style, rng=rng)
    fg = get_foreground_color(style)
    axis_color = get_axis_text_color(style, rng=rng)

    n_color_keys = list(dict.fromkeys(plot_df["color_group"].tolist()))
    n_shape_keys = list(dict.fromkeys(plot_df["shape_group"].tolist()))
    palette = build_palette(len(n_color_keys), style, rng=rng)
    shape_sequence = build_shape_sequence(style, max(1, len(n_shape_keys)))
    shape_values = [get_altair_shape(s) for s in shape_sequence]

    legend_orient, _ = legend_position_altair(style)
    legend = None
    if int(style.get("legend_present", 0)) == 1 and legend_orient:
        legend = alt.Legend(
            orient=legend_orient,
            title="Group" if int(style.get("legend_title_size", 0)) == 1 else None,
            labelColor=get_legend_text_color(style) or fg,
            titleColor=get_title_color(style, rng=rng),
        )

    color_scale = alt.Scale(domain=n_color_keys, range=palette)
    shape_scale = alt.Scale(domain=n_shape_keys, range=shape_values)

    grid_code = int(style.get("gridlines", 0))
    x_grid = grid_code in {1, 3}
    y_grid = grid_code in {1, 2, 3}
    is_dashed = int(style.get("gridline_color", 0)) == 4
    gridcolor = get_gridline_color(style)

    # Use actual data range
    x_pad = (float(plot_df["x"].max()) - float(plot_df["x"].min())) * 0.05 or 1.0
    y_pad = (float(plot_df["y"].max()) - float(plot_df["y"].min())) * 0.05 or 1.0
    x_lo = float(plot_df["x"].min()) - x_pad
    x_hi = float(plot_df["x"].max()) + x_pad
    y_lo = float(plot_df["y"].min()) - y_pad
    y_hi = float(plot_df["y"].max()) + y_pad

    x_tick = (x_hi - x_lo) / 10
    y_tick = (y_hi - y_lo) / 10

    x_axis = alt.Axis(
        labelColor=axis_color, titleColor=axis_color,
        grid=x_grid, gridColor=gridcolor, gridOpacity=0.8,
        gridDash=[4, 4] if is_dashed else [],
        tickMinStep=x_tick,
        labelAngle=0,
        zindex=0, # gridlines behind everything
    )
    y_axis = alt.Axis(
        labelColor=axis_color, titleColor=axis_color,
        grid=y_grid, gridColor=gridcolor, gridOpacity=0.8,
        gridDash=[4, 4] if is_dashed else [],
        tickMinStep=y_tick,
        zindex=0, #gridlines behind everything
    )

    base = alt.Chart(plot_df).encode(
        x=alt.X("x:Q", title=x_label, scale=alt.Scale(domain=[x_lo, x_hi]), axis=x_axis),
        y=alt.Y("y:Q", title=y_label, scale=alt.Scale(domain=[y_lo, y_hi]), axis=y_axis),
        color=alt.Color("color_group:N", scale=color_scale, legend=legend),
        shape=alt.Shape("shape_group:N", scale=shape_scale, legend=None),
        tooltip=["point_id", "group", "x", "y"],
    )

    outline_only = int(style.get("point_shape_mode", 0)) == 3
    point_opacity = 0.4 if int(style.get("point_shape_mode", 0)) == 5 else 1.0

    if outline_only:
        points = base.mark_point(
            filled=True,
            size=75,
            fillOpacity=1.0,
            stroke="black",
            strokeWidth=0.8,
        )
    else:
        points = base.mark_point(
            filled=True,
            size=75,
            fillOpacity=point_opacity,
        )


    layers = [points]

    label_df = plot_df[plot_df["show_label"]].copy()
    if int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) == 1 and len(label_df) > 0:
        labels = alt.Chart(label_df).mark_text(dx=8, dy=-8, color=get_label_color(style, rng=rng), fontSize=11).encode(
            x="x:Q", y="y:Q", text="label:N"
        )
        layers.append(labels)

    # Build title
    title_params = alt.TitleParams(text="", subtitle="")
    if int(style.get("title_present", 0)) == 1 and title:
        sub = subtitle if int(style.get("subtitle_present", 0)) == 1 and subtitle else ""
        title_params = alt.TitleParams(
            text=title,
            subtitle=sub,
            anchor=get_altair_title_anchor(style),
            fontSize=get_title_fontsize(style, rng=rng),
            color=get_title_color(style, rng=rng),
            subtitleColor=fg,
        )

    # chart_outline
    outline = int(style.get("chart_outline", 1))
    view_stroke = fg if outline == 2 else "transparent"

    chart = (
        alt.layer(*layers)
        .properties(width=640, height=420, title=title_params)
        .configure(
            background=bg if bg != "none" else alt.Undefined,
            axis=alt.AxisConfig(
                domainColor=fg, tickColor=fg,
                labelColor=axis_color, titleColor=axis_color,
            ),
            legend=alt.LegendConfig(
                strokeColor=fg if int(style.get("legend_outline", 0)) == 1 else "transparent",
                padding=6,
                fillColor=get_legend_fill_color(style, rng=rng) or "transparent",
            ),
            view=alt.ViewConfig(stroke=view_stroke),
        )
    )

    return chart

In [129]:
def render_scatter_matplotlib(plot_df: pd.DataFrame, reg_df, title: str, subtitle: str | None, style: dict, rng: np.random.Generator | None = None, x_label: str = "X value", y_label: str = "Y value"):
    bg = get_background_color(style, rng=rng)
    fig, ax = plt.subplots(figsize=(8.3, 5.4))
    fig.patch.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_axisbelow(True)

    color_keys = list(dict.fromkeys(plot_df["color_group"].tolist()))
    shape_keys = list(dict.fromkeys(plot_df["shape_group"].tolist()))
    palette = build_palette(len(color_keys), style, rng=rng)
    color_map = {k: palette[i] for i, k in enumerate(color_keys)}
    shape_sequence = build_shape_sequence(style, max(1, len(shape_keys)))
    shape_map = {k: get_mpl_marker(shape_sequence[i % len(shape_sequence)]) for i, k in enumerate(shape_keys)}
    
    point_opacity = 0.4 if int(style.get("point_shape_mode", 0)) == 5 else 0.9
    outline_only = int(style.get("point_shape_mode", 0)) == 3

    for (_, row) in plot_df.iterrows():
        ax.scatter(
            row["x"], row["y"],
            c="none" if outline_only else color_map[row["color_group"]],
            marker=shape_map[row["shape_group"]],
            s=60, 
            edgecolors="black" if outline_only else "none",
            linewidths=1.5 if outline_only else 0.0,
            zorder=3,
            alpha=point_opacity,
        )

    if int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) == 1:
        for (_, row) in plot_df[plot_df["show_label"]].iterrows():
            ax.annotate(row["label"], (row["x"], row["y"]), xytext=(5, 5),
                        textcoords="offset points", fontsize=9, color=get_label_color(style, rng=rng))

    reg_line = int(style.get("regression_line", 0))
    if reg_df is not None and len(reg_df) > 0 and reg_line > 0:
        bright_colors = ["#ff006e", "#fb5607", "#8338ec", "#3a86ff"]
        reg_line_format = int(style.get("regression_line_format", 1))
        reg_line_color  = int(style.get("regression_line_color", 2))
        for i, group in enumerate(list(dict.fromkeys(reg_df["group"].tolist()))):
            gdf = reg_df[reg_df["group"] == group]
            dash = "--" if reg_line_format in {2, 3} else "-"
            if reg_line_color == 3:
                color = "#bbbbbb"
            elif reg_line_color == 4:
                color = bright_colors[i % len(bright_colors)]
            elif reg_line_color == 1:
                color = "#444444"
            else:
                color = color_map.get(group, palette[i % len(palette)])
            ax.plot(gdf["x"], gdf["yhat"], linestyle=dash, linewidth=2, color=color)
            if int(style.get("regression_confidence_band", 0)) == 1:
                ax.fill_between(gdf["x"], gdf["y_lower"], gdf["y_upper"], color=color, alpha=0.15)
            if int(style.get("regression_confidence_bounds", 0)) == 1:
                ax.plot(gdf["x"], gdf["y_upper"], linestyle=":", linewidth=1.2, color=color, alpha=0.7)
                ax.plot(gdf["x"], gdf["y_lower"], linestyle=":", linewidth=1.2, color=color, alpha=0.7)

    axis_color = get_axis_text_color(style, rng=rng)

    # Use actual data range with padding
    x_pad = (float(plot_df["x"].max()) - float(plot_df["x"].min())) * 0.05 or 1.0
    y_pad = (float(plot_df["y"].max()) - float(plot_df["y"].min())) * 0.05 or 1.0
    x_lo = float(plot_df["x"].min()) - x_pad
    x_hi = float(plot_df["x"].max()) + x_pad
    y_lo = float(plot_df["y"].min()) - y_pad
    y_hi = float(plot_df["y"].max()) + y_pad
    x_step = (x_hi - x_lo) / 10
    y_step = (y_hi - y_lo) / 10

    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel(x_label, color=axis_color)
    ax.set_ylabel(y_label, color=axis_color)
    ax.tick_params(colors=axis_color)
    ax.set_xticks(np.arange(x_lo, x_hi + x_step * 0.01, x_step))
    ax.set_yticks(np.arange(y_lo, y_hi + y_step * 0.01, y_step))

    axis_text_orient = int(style.get("axis_text_orientation", 0))
    for label in ax.get_xticklabels():
        label.set_rotation(0)
        label.set_ha("center")

    apply_gridline_state_mpl(ax, style)
    apply_outline_mpl(ax, fig, style, rng=rng)

    if int(style.get("title_present", 0)) == 1:
        _, halign = get_title_alignment(style)
        title_text = title if int(style.get("subtitle_present", 0)) == 0 else f"{title}\n{subtitle}"
        ax.set_title(title_text, loc=halign, color=get_title_color(style, rng=rng),
                     fontsize=get_title_fontsize(style, rng=rng), pad=12)

    legend_orient_code = int(style.get("legend_orientation", 0))
    if int(style.get("legend_present", 0)) == 1 and legend_orient_code != 0:
        entry = LEGEND_ORIENTATIONS.get(legend_orient_code)
        if entry:
            _, loc, bbox, *_ = entry
            handles, labels = [], []
            for group in list(dict.fromkeys(plot_df["group"].tolist())):
                sub = plot_df[plot_df["group"] == group].iloc[0]
                handles.append(plt.Line2D([0], [0], marker=shape_map[sub["shape_group"]],
                                color="w", markerfacecolor=color_map[sub["color_group"]], markersize=8))
                labels.append(group)
            legend_fill = int(style.get("legend_fill", 0))
            has_outline = int(style.get("legend_outline", 0)) == 1
            legend_kwargs = {"frameon": has_outline or legend_fill in {1, 3}}
            fill_color = get_legend_fill_color(style, rng=rng)
            if fill_color:
                legend_kwargs["facecolor"] = fill_color
            ncol = len(labels) if legend_orient_code in {3, 4} else 1
            legend = ax.legend(handles, labels,
                                title="Group" if int(style.get("legend_title_size", 0)) == 1 else None,
                                loc=loc, bbox_to_anchor=bbox, ncol=ncol, **legend_kwargs)
            frame = legend.get_frame()
            if has_outline:
                frame.set_edgecolor(get_foreground_color(style))
                frame.set_linewidth(1.2)
            else:
                frame.set_edgecolor("none")
            txt_color = get_legend_text_color(style, rng=rng)
            if txt_color:
                for txt in legend.get_texts():
                    txt.set_color(txt_color)

    if legend_orient_code == 4:
        fig.tight_layout()
        plt.subplots_adjust(bottom=0.25)
    elif legend_orient_code in {3, 5, 6}:
        fig.tight_layout()
        plt.subplots_adjust(top=0.82)
    elif legend_orient_code == 1:
        fig.tight_layout()
        plt.subplots_adjust(left=0.22)
    elif legend_orient_code == 2:
        fig.tight_layout()
        plt.subplots_adjust(right=0.75)
    else:
        fig.tight_layout()
    return fig


In [130]:
def render_scatter_seaborn(plot_df: pd.DataFrame, reg_df, title: str, subtitle: str | None, style: dict, rng: np.random.Generator | None = None, x_label: str = "X value", y_label: str = "Y value"):
    if sns is None:
        raise ImportError("seaborn is not installed")

    bg = get_background_color(style, rng=rng)
    fig, ax = plt.subplots(figsize=(8.3, 5.4))
    fig.patch.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_axisbelow(True)

    color_keys = list(dict.fromkeys(plot_df["color_group"].tolist()))
    palette_values = build_palette(len(color_keys), style, rng=rng)
    color_map = {k: palette_values[i] for i, k in enumerate(color_keys)}

    point_opacity = 0.4 if int(style.get("point_shape_mode", 0)) == 5 else 0.9

    sns.scatterplot(
        data=plot_df, x="x", y="y", hue="color_group",
        style="shape_group" if int(style.get("point_shape_mode", 0)) in {2, 4} else None,
        palette=color_map, s=70, edgecolor="none", ax=ax, legend=False,
        zorder=3,
        alpha=point_opacity,
    )

    if int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) == 1:
        for _, row in plot_df[plot_df["show_label"]].iterrows():
            ax.annotate(row["label"], (row["x"], row["y"]), xytext=(5, 5),
                        textcoords="offset points", fontsize=9, color=get_label_color(style, rng=rng))

    reg_line = int(style.get("regression_line", 0))
    if reg_df is not None and len(reg_df) > 0 and reg_line > 0:
        bright_colors = ["#ff006e", "#fb5607", "#8338ec", "#3a86ff"]
        reg_line_format = int(style.get("regression_line_format", 1))
        reg_line_color  = int(style.get("regression_line_color", 2))
        for i, group in enumerate(list(dict.fromkeys(reg_df["group"].tolist()))):
            gdf = reg_df[reg_df["group"] == group]
            dash = "--" if reg_line_format in {2, 3} else "-"
            if reg_line_color == 3:
                color = "#bbbbbb"
            elif reg_line_color == 4:
                color = bright_colors[i % len(bright_colors)]
            elif reg_line_color == 1:
                color = "#444444"
            else:
                color = color_map.get(group, palette_values[i % len(palette_values)])
            ax.plot(gdf["x"], gdf["yhat"], linestyle=dash, linewidth=2, color=color)
            if int(style.get("regression_confidence_band", 0)) == 1:
                ax.fill_between(gdf["x"], gdf["y_lower"], gdf["y_upper"], color=color, alpha=0.15)
            if int(style.get("regression_confidence_bounds", 0)) == 1:
                ax.plot(gdf["x"], gdf["y_upper"], linestyle=":", linewidth=1.2, color=color, alpha=0.7)
                ax.plot(gdf["x"], gdf["y_lower"], linestyle=":", linewidth=1.2, color=color, alpha=0.7)

    axis_color = get_axis_text_color(style, rng=rng)

    x_pad = (float(plot_df["x"].max()) - float(plot_df["x"].min())) * 0.05 or 1.0
    y_pad = (float(plot_df["y"].max()) - float(plot_df["y"].min())) * 0.05 or 1.0
    x_lo = float(plot_df["x"].min()) - x_pad
    x_hi = float(plot_df["x"].max()) + x_pad
    y_lo = float(plot_df["y"].min()) - y_pad
    y_hi = float(plot_df["y"].max()) + y_pad
    x_step = (x_hi - x_lo) / 10
    y_step = (y_hi - y_lo) / 10

    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel(x_label, color=axis_color)
    ax.set_ylabel(y_label, color=axis_color)
    ax.tick_params(colors=axis_color)
    ax.set_xticks(np.arange(x_lo, x_hi + x_step * 0.01, x_step))
    ax.set_yticks(np.arange(y_lo, y_hi + y_step * 0.01, y_step))

    for label in ax.get_xticklabels():
        label.set_rotation(0)
        label.set_ha("center")

    apply_gridline_state_mpl(ax, style, rng=rng)
    apply_outline_mpl(ax, fig, style, rng=rng)

    if int(style.get("title_present", 0)) == 1:
        _, halign = get_title_alignment(style)
        title_text = title if int(style.get("subtitle_present", 0)) == 0 else f"{title}\n{subtitle}"
        ax.set_title(title_text, loc=halign, color=get_title_color(style, rng=rng),
                     fontsize=get_title_fontsize(style, rng=rng), pad=12)

    legend_orient_code = int(style.get("legend_orientation", 0))
    if int(style.get("legend_present", 0)) == 1 and legend_orient_code != 0:
        entry = LEGEND_ORIENTATIONS.get(legend_orient_code)
        if entry:
            _, loc, bbox, *_ = entry
            handles, labels = [], []
            for group in list(dict.fromkeys(plot_df["group"].tolist())):
                sub = plot_df[plot_df["group"] == group].iloc[0]
                handles.append(plt.Line2D([0], [0], marker="o", color="w",
                                markerfacecolor=color_map[sub["color_group"]], markersize=8))
                labels.append(group)
            has_outline = int(style.get("legend_outline", 0)) == 1
            legend = ax.legend(handles, labels,
                      title="Group" if int(style.get("legend_title_size", 0)) == 1 else None,
                      loc=loc, bbox_to_anchor=bbox,
                      frameon=has_outline)
            frame = legend.get_frame()
            if has_outline:
                frame.set_edgecolor(get_foreground_color(style))
                frame.set_linewidth(1.2)
            else:
                frame.set_edgecolor("none")

    if legend_orient_code == 4:
        fig.tight_layout()
        plt.subplots_adjust(bottom=0.25)
    elif legend_orient_code in {3, 5, 6}:
        fig.tight_layout()
        plt.subplots_adjust(top=0.82)
    elif legend_orient_code == 1:
        fig.tight_layout()
        plt.subplots_adjust(left=0.22)
    elif legend_orient_code == 2:
        fig.tight_layout()
        plt.subplots_adjust(right=0.75)
    else:
        fig.tight_layout()
    return fig

In [131]:
def render_scatter_plotly(plot_df: pd.DataFrame, reg_df, title: str, subtitle: str | None, style: dict, rng: np.random.Generator | None = None, x_label: str = "X value", y_label: str = "Y value"):
    color_keys = list(dict.fromkeys(plot_df["color_group"].tolist()))
    palette = build_palette(len(color_keys), style, rng=rng)
    color_map = {k: palette[i] for i, k in enumerate(color_keys)}
    shape_keys = list(dict.fromkeys(plot_df["shape_group"].tolist()))
    shape_sequence = build_shape_sequence(style, max(1, len(shape_keys)))
    symbol_map = {k: get_plotly_symbol(shape_sequence[i % len(shape_sequence)]) for i, k in enumerate(shape_keys)}

    show_legend = int(style.get("legend_present", 0)) == 1
    show_labels = int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) == 1
    
    outline_only = int(style.get("point_shape_mode", 0)) == 3
    point_opacity = 0.4 if int(style.get("point_shape_mode", 0)) == 5 else 0.9

    fig = go.Figure()
    for group in list(dict.fromkeys(plot_df["group"].tolist())):
        gdf = plot_df[plot_df["group"] == group].copy()
        if show_labels:
            if int(style.get("direct_labels", 0)) == 2:
                custom_text = np.where(gdf["show_label"], gdf["label"], "")
            else:
                custom_text = gdf["label"]
            mode = "markers+text"
        else:
            custom_text = [""] * len(gdf)
            mode = "markers"
        fig.add_trace(go.Scatter(
        x=gdf["x"], y=gdf["y"],
        mode=mode,
        text=custom_text,
        textposition="top center",
        textfont=dict(color=get_label_color(style, rng=rng)),
        name=group,
        showlegend=show_legend,
        marker=dict(
            color=color_map[gdf.iloc[0]["color_group"]],
            size=10,
            opacity=point_opacity,
            symbol=symbol_map[gdf.iloc[0]["shape_group"]],
            line=dict(
                color="rgba(0,0,0,0.4)" if outline_only else "rgba(0,0,0,0)",
                width=0.8 if outline_only else 0,
            ),
        ),
        hovertemplate="%{text}<br>x=%{x:.2f}<br>y=%{y:.2f}<extra>" + group + "</extra>",
    ))



    bg = get_background_color(style, rng=rng)
    paper_bg = "rgba(0,0,0,0)" if bg == "none" else bg
    fg = get_foreground_color(style)
    axis_color = get_axis_text_color(style, rng=rng)

    x_pad = (float(plot_df["x"].max()) - float(plot_df["x"].min())) * 0.05 or 1.0
    y_pad = (float(plot_df["y"].max()) - float(plot_df["y"].min())) * 0.05 or 1.0
    x_lo = float(plot_df["x"].min()) - x_pad
    x_hi = float(plot_df["x"].max()) + x_pad
    y_lo = float(plot_df["y"].min()) - y_pad
    y_hi = float(plot_df["y"].max()) + y_pad
    x_tick = (x_hi - x_lo) / 10
    y_tick = (y_hi - y_lo) / 10

    title_text = None
    if int(style.get("title_present", 0)) == 1:
        title_text = title if int(style.get("subtitle_present", 0)) == 0 else f"{title}<br><sup>{subtitle}</sup>"
    x_pos, x_anchor = get_plotly_title_anchor(style)

    grid_code = int(style.get("gridlines", 0))
    gridcolor = get_gridline_color(style)
    gridcolor, grid_ls = get_gridline_style(style, rng=rng)
    dash_style = "dot" if grid_ls == "dashed" else "solid"

    has_outline = int(style.get("legend_outline", 0)) == 1
    legend_bgcolor = get_legend_fill_color(style, rng=rng) or "rgba(0,0,0,0)"
    legend_text_color = get_legend_text_color(style, rng=rng) or fg

    fig.update_layout(
        title=dict(text=title_text, x=x_pos, xanchor=x_anchor,
                   font=dict(color=get_title_color(style, rng=rng),
                             size=get_title_fontsize(style, rng=rng))) if title_text else None,
        plot_bgcolor=paper_bg,
        paper_bgcolor=paper_bg,
        font=dict(color=fg),
        xaxis=dict(
            title=x_label, range=[x_lo, x_hi],
            dtick=x_tick,
            tickangle=0,
            color=axis_color,
            zeroline=False,
            layer="below traces",
        ),
        yaxis=dict(
            title=y_label, range=[y_lo, y_hi],
            dtick=y_tick,
            color=axis_color,
            zeroline=False,
            layer="below traces",
        ),
        showlegend=show_legend,
        legend=dict(
            bordercolor=fg if has_outline else "rgba(0,0,0,0)",
            borderwidth=1.5 if has_outline else 0,
            bgcolor=legend_bgcolor,
            font=dict(color=legend_text_color),
        ),
        width=760,
        height=500,
    )

    if grid_code == 0:
        fig.update_xaxes(showgrid=False, zeroline=False)
        fig.update_yaxes(showgrid=False, zeroline=False)
    elif grid_code == 2:
        fig.update_xaxes(showgrid=False, zeroline=False)
        fig.update_yaxes(showgrid=True, gridcolor=gridcolor, gridwidth=1, griddash=dash_style, zeroline=False)
    else:
        fig.update_xaxes(showgrid=True, gridcolor=gridcolor, gridwidth=1, griddash=dash_style, zeroline=False)
        fig.update_yaxes(showgrid=True, gridcolor=gridcolor, gridwidth=1, griddash=dash_style, zeroline=False)

    outline = int(style.get("chart_outline", 1))
    if outline == 0:
        fig.update_xaxes(showline=False, zeroline=False)
        fig.update_yaxes(showline=False, zeroline=False)
    elif outline == 1:
        fig.update_xaxes(showline=True, linewidth=1, linecolor=fg, mirror=False, zeroline=False)
        fig.update_yaxes(showline=True, linewidth=1, linecolor=fg, mirror=False, zeroline=False)
    else:
        fig.update_xaxes(showline=True, linewidth=1, linecolor=fg, mirror=True, zeroline=False)
        fig.update_yaxes(showline=True, linewidth=1, linecolor=fg, mirror=True, zeroline=False)

    if int(style.get("image_outline", 0)) == 1:
        fig.update_layout(shapes=[dict(type="rect", xref="paper", yref="paper",
                                       x0=0, y0=0, x1=1, y1=1,
                                       line=dict(color=fg, width=1))])
    return fig

## 5. Saving and generation


In [132]:
def new_chart_id(prefix: str = "scatter") -> str:
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"


def save_metadata(meta: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)


def ensure_output_dirs(out_root: Path) -> None:
    if CLEAR_OUTPUT and out_root.exists():
        shutil.rmtree(out_root)
    for sub in SUBDIRS:
        for lib in LIBRARIES:
            (out_root / sub / lib).mkdir(parents=True, exist_ok=True)


# def save_altair_svg(chart, out_path: Path) -> None:
#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     chart.save(str(out_path), format="svg")

def install_package(pip_name: str):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

def ensure_altair_png_support():
    try:
        import vl_convert  # noqa: F401
    except Exception:
        install_package("vl-convert-python")

def save_altair_png(chart, path: Path) -> Path:
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    ensure_altair_png_support()
    chart.save(str(path))
    return path


def install_package(pip_name: str):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

def save_plotly_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(out_path), format="png", scale=2)


def save_matplotlib_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200, bbox_inches="tight",
                transparent=(fig.get_facecolor()[-1] == 0
                             if hasattr(fig.get_facecolor(), "__len__") else False))
    plt.close(fig)

def generate_scatter(
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    param_stats: dict,
    datasets: dict | None = None,
    max_tries: int = 25,
) -> dict:
    rng = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("scatter")

    for attempt in range(1, max_tries + 1):
        style = harmonize_style(sample_style(rng, param_stats))

        # ── Sample data ────────────────────────────────────────────────────
        plot_df = None
        context = None

        if datasets:
            # Pick a random available dataset
            ds_names = list(datasets.keys())
            ds_name  = ds_names[int(rng.integers(0, len(ds_names)))]
            df       = datasets[ds_name]
            spec     = DATASET_REGISTRY[ds_name]
            result   = sample_scatter_data_from_df(df, spec, rng, style)
            if result is not None:
                plot_df, context = result
                dataset_source = ds_name
            else:
                print(f"[seed {rng_seed}] Real data sampling failed → synthetic fallback")

        # Fallback to synthetic data if real data sampling failed
        if plot_df is None:
            plot_df, _, context = sample_scatter_data(rng=rng, style=style)

        # Axis labels come from real data context
        x_label = context.get("x_label", "X value")
        y_label = context.get("y_label", "Y value")

        title, subtitle = make_title(context, style=style)

        table_path = out_root / "tables" / library / f"{chart_id}.csv"
        meta_path  = out_root / "metadata" / library / f"{chart_id}.json"
        plot_df.to_csv(table_path, index=False)

        # Pass x_label and y_label to renderers
        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            chart = render_scatter_altair(plot_df, None, title, subtitle, style,
                                          x_label=x_label, y_label=y_label)
            save_altair_png(chart, image_path)
        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_scatter_matplotlib(plot_df, None, title, subtitle, style,
                                             rng=rng, x_label=x_label, y_label=y_label)
            save_matplotlib_png(fig, image_path)
        elif library == "seaborn":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_scatter_seaborn(plot_df, None, title, subtitle, style,
                                          rng=rng, x_label=x_label, y_label=y_label)
            save_matplotlib_png(fig, image_path)
        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_scatter_plotly(plot_df, None, title, subtitle, style,
                                         x_label=x_label, y_label=y_label)
            save_plotly_png(fig, image_path)
        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id":      chart_id,
            "chart_type":    "scatter",
            "library":       library,
            "dataset_source": dataset_source,
            "image_path":    str(image_path),
            "table_path":    str(table_path),
            "data_context":  context,
            "style":         style,
            "created_utc":   datetime.utcnow().isoformat() + "Z",
            "rng_seed":      rng_seed,
        }
        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate scatter after {max_tries} attempts.")


def generate_batch(
    out_root: Path,
    dataset_source: str,
    generation_plan: dict,
    param_stats: dict,
    datasets: dict | None = None,
    start_seed: int = 1000,
) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []
    seed = start_seed
    for library, n in generation_plan.items():
        for _ in range(n):
            metas.append(generate_scatter(
                out_root=out_root,
                dataset_source=dataset_source,
                library=library,
                rng_seed=seed,
                param_stats=param_stats,
                datasets=datasets,
            ))
            seed += 1
    return metas




# def generate_batch(
#     out_root: Path,
#     dataset_source: str,
#     generation_plan: dict,
#     param_stats: dict,
#     datasets: dict | None = None,
#     start_seed: int = 1000,
# ) -> list[dict]:
#     ensure_output_dirs(out_root)
#     metas = []
#     seed = start_seed
#     for library, n in generation_plan.items():
#         for _ in range(n):
#             metas.append(generate_scatter(
#                 out_root=out_root,
#                 dataset_source=dataset_source,
#                 library=library,
#                 rng_seed=seed,
#                 param_stats=param_stats,
#                 datasets=datasets,
#             ))
#             seed += 1
#     return metas


In [133]:
ensure_output_dirs(OUT_ROOT)

weights = OBSERVED_WEIGHTS

generation_plan = {
    "altair":     10,
    "matplotlib": 10,
    "seaborn":    10,
    "plotly":     10,
}

metas = generate_batch(
    out_root=OUT_ROOT,
    dataset_source="warehouse_retail",
    generation_plan=generation_plan,
    param_stats=weights,
    datasets=DATASETS,   # pass real datasets here; set to None for synthetic only
    start_seed=1000,
)

pd.DataFrame(metas)[["chart_id", "library", "dataset_source", "image_path"]].head()


KeyboardInterrupt: 

# TESTING


### Smoke test, synthetic

### Smoke test, real data

### Legend

In [ ]:
# from pathlib import Path
# import re

# TEST_LIBRARIES = ["matplotlib", "seaborn", "altair", "plotly"]
# SCATTER_TESTING_ROOT = PROJECT / "testing" / "scatter_testing"
# shutil.rmtree(SCATTER_TESTING_ROOT, ignore_errors=True)

# BASE_STYLE = {param: max(codes, key=codes.get) for param, codes in SAMPLING_WEIGHTS.items()}

# # For each parameter, define which other params need to be forced
# # so the tested parameter is actually visible in the output
# PREREQS = {
#     "title_location":        {"title_present": 1},
#     "title_color":           {"title_present": 1},
#     "title_size":            {"title_present": 1},
#     "subtitle_present":      {"title_present": 1},
#     "legend_title_size":     {"legend_present": 1, "legend_orientation": 2, "n_groups": 3, "n_colors": 3},
#     "legend_title_color":    {"legend_present": 1, "legend_orientation": 2, "legend_title_size": 1, "n_groups": 3, "n_colors": 3},
#     "legend_text_color":     {"legend_present": 1, "legend_orientation": 2, "n_groups": 3, "n_colors": 3},
#     "legend_outline":        {"legend_present": 1, "legend_orientation": 2, "n_groups": 3, "n_colors": 3},
#     "legend_fill":           {"legend_present": 1, "legend_orientation": 2, "n_groups": 3, "n_colors": 3},
#     "legend_orientation":    {"legend_present": 1, "n_groups": 3, "n_colors": 3},
#     "label_content":         {"direct_labels": 1},
#     "label_color":           {"direct_labels": 1, "label_content": 1},
#     "gridline_color":        {"gridlines": 1},
#     "scatter_orientation":   {"n_groups": 3},
#     "point_shape_mode":      {"n_groups": 3, "n_shapes": 3},
#     "n_shapes":              {"point_shape_mode": 2, "n_groups": 3},
#     "n_colors":              {"n_groups": 3},
#     "palette_type":          {"n_colors": 3, "n_groups": 3},
#     "minor_ticks_x":         {"gridlines": 0},
#     "minor_ticks_y":         {"gridlines": 0},
#     "x_tick_step":           {"x_scale": 0},
#     "y_tick_step":           {"y_scale": 0},
# }

# def ensure_scatter_testing_dirs(out_root: Path) -> None:
#     for sub in SUBDIRS:
#         for lib in LIBRARIES:
#             (out_root / sub / lib).mkdir(parents=True, exist_ok=True)

# def sanitize_slug(text) -> str:
#     text = re.sub(r"[^a-zA-Z0-9._-]+", "_", str(text))
#     return text.strip("_").lower()
# def render_single_test_case(param_name: str, code: int, library: str, out_root: Path) -> dict:
#     style = {**BASE_STYLE}
#     style.update(PREREQS.get(param_name, {}))
#     style[param_name] = code

#     if param_name not in {"title_present", "title_location", "title_color", "title_size"}:
#         style["title_present"] = 1
#         style["title_location"] = 1
#         style["title_color"] = 0
#         style["title_size"] = 0

#     style = harmonize_style(style)   # ← ensures all dependent params are consistent

#     rng = np.random.default_rng(2026)
#     plot_df, reg_df, context = sample_scatter_data(rng, style)

#     title = f"{param_name.replace('_', ' ')} = {code}"
#     subtitle = make_subtitle(context)
#     case_name = f"{param_name}__code_{code}"
#     chart_id = sanitize_slug(case_name)

#     table_path = out_root / "tables" / library / f"{chart_id}.csv"
#     meta_path  = out_root / "metadata" / library / f"{chart_id}.json"
#     plot_df.to_csv(table_path, index=False)

#     if library == "altair":
#         image_path = out_root / "images" / library / f"{chart_id}.svg"
#         chart = render_scatter_altair(plot_df, reg_df, title, subtitle, style)
#         save_altair_svg(chart, image_path)
#     elif library == "matplotlib":
#         image_path = out_root / "images" / library / f"{chart_id}.png"
#         fig = render_scatter_matplotlib(plot_df, reg_df, title, subtitle, style)
#         save_matplotlib_png(fig, image_path)
#     elif library == "seaborn":
#         image_path = out_root / "images" / library / f"{chart_id}.png"
#         fig = render_scatter_seaborn(plot_df, reg_df, title, subtitle, style)
#         save_matplotlib_png(fig, image_path)
#     else:
#         image_path = out_root / "images" / library / f"{chart_id}.png"
#         fig = render_scatter_plotly(plot_df, reg_df, title, subtitle, style)
#         save_plotly_png(fig, image_path)

#     meta = {
#         "chart_id": chart_id, "library": library,
#         "param_tested": param_name, "code_tested": code,
#         "style": style,
#         "image_path": str(image_path), "table_path": str(table_path),
#     }
#     save_metadata(meta, meta_path)
#     return meta


# ensure_scatter_testing_dirs(SCATTER_TESTING_ROOT)
# all_test_metas = []

# for param_name, code_probs in SAMPLING_WEIGHTS.items():
#     for code in sorted(code_probs.keys()):
#         for library in TEST_LIBRARIES:
#             all_test_metas.append(
#                 render_single_test_case(param_name, code, library, SCATTER_TESTING_ROOT)
#             )

# pd.DataFrame(all_test_metas)[["chart_id", "library", "param_tested", "code_tested", "image_path"]].head(20)